In [2]:
"""
DirectLiNGAM Graph & Gamma Discovery — Soil + Yield
=====================================================
Runs DirectLiNGAM on both datasets to:
  (A) Recover the causal graph (compare vs known true DAG)
  (B) Extract γᵢ weights from the adjacency matrix row for the target
  (C) Compare recovered γ to ground-truth γ using Pearson r, Spearman r, RMSE
  (D) Run Causal SHAP with the LiNGAM-recovered γ and compare to:
        - true γ baseline
        - PC-discovered γ (uniform proxy, since PC gives CPDAG not weights)
        - standard SHAP baseline

Also runs:
  (E) Jarque-Bera non-Gaussianity test on residuals (prerequisite check for LiNGAM)
  (F) DAGMA (linear) as fallback comparison

Install requirements:
    pip install lingam dagma causallearn xgboost scipy scikit-learn networkx pandas numpy

Outputs:
    lingam_results_soil.csv
    lingam_results_yield.csv
    (printed graph comparison tables and metric summaries)
"""
from gettext import install

import pip


!pip install lingam dagma
import time
import itertools
import math
import warnings
from typing import Dict, List, Set, Tuple

import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import pearsonr, spearmanr, jarque_bera
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import networkx as nx
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

# ── Try importing causal discovery libraries ──────────────────────
try:
    import lingam
    HAS_LINGAM = True
    print("lingam available")
except ImportError:
    HAS_LINGAM = False
    print("x lingam not installed — run: pip install lingam")

try:
    from dagma.linear import DagmaLinear
    HAS_DAGMA = True
    print("dagma available")
except ImportError:
    HAS_DAGMA = False
    print("x dagma not installed — run: pip install dagma")

# Config
CONFIG = {
    'SOIL_DATA_PATH':         '../../../datasets/data_complete.csv',
    'SOIL_GROUND_TRUTH_PATH': '../../../datasets/ground_truth.csv',
    'YIELD_DATA_PATH':        '../../../datasets/yield_regression_preprocessed.npz',
    'TEST_SIZE':     0.20,
    'RANDOM_SEED':   42,
    # Causal SHAP settings
    'YIELD_MAX_PERM': 500,
    'YIELD_M':        128,
    'YIELD_K':        100,
    'SOIL_M':         128,
    'SOIL_K':         100,
}

SOIL_FEATURES   = ['Organic Carbon','Nitrogen','Phosphorus','Potassium',
                    'pH','Zinc','Electrical Conductivity']
SOIL_TARGET     = 'SPI_CONTINUOUS'
SOIL_COL_MAP    = {'OC':'Organic Carbon','N':'Nitrogen','P':'Phosphorus',
                   'K':'Potassium','PH':'pH','ZN':'Zinc','EC':'Electrical Conductivity'}
SOIL_TRUE_EDGES = {('Organic Carbon','pH'), ('pH','Zinc')}
# True causal edges to target (all 7 features -> SPI in true DGP)
SOIL_TARGET_EDGES = {'Organic Carbon','Nitrogen','Phosphorus','Potassium',
                     'pH','Zinc','Electrical Conductivity'}

YIELD_FEATURES  = ['Nitrogen','Irrigation','Precipitation','Herbicide']
YIELD_TARGET    = 'Yield'
YIELD_TRUE_EDGES = set()  # star DAG - no inter-feature edges
YIELD_TRUE_TRANSLOG = {
    'betaN':0.35,'betaI':0.50,'betaP':0.25,'betaH':0.05,
    'betaNN':-0.02,'betaII':-0.03,'betaPP':-0.01,'betaHH':-0.005,
    'betaNI':0.04,'betaNP':0.02,'betaNH':0.01,
    'betaIP':0.03,'betaIH':0.01,'betaPH':0.005,
}

np.random.seed(CONFIG['RANDOM_SEED'])
rng = np.random.default_rng(CONFIG['RANDOM_SEED'])


# Ground-truth gamma
def compute_soil_gamma(path: str) -> Dict[str, float]:
    gt  = pd.read_csv(path)
    net = pd.DataFrame({
        'Organic Carbon':          gt['OC_base']  - gt['OC_penalty'],
        'Nitrogen':                gt['N_base']   - gt['N_penalty'],
        'Phosphorus':              gt['P_base']   - gt['P_penalty'],
        'Potassium':               gt['K_base']   - gt['K_penalty'],
        'pH':                     -gt['PH_penalty'],
        'Zinc':                    gt['ZN_base']  - gt['ZN_penalty'],
        'Electrical Conductivity': -gt['EC_penalty'],
    })
    m = net.abs().mean()
    return (m / m.sum()).to_dict()


def compute_yield_gamma(X_raw: pd.DataFrame) -> Dict[str, float]:
    c   = YIELD_TRUE_TRANSLOG
    lm  = {f: float(np.log(X_raw[f]).mean()) for f in YIELD_FEATURES}
    lnN, lnI, lnP, lnH = lm['Nitrogen'], lm['Irrigation'], lm['Precipitation'], lm['Herbicide']
    e = {
        'Nitrogen':      c['betaN']  + c['betaNN']*lnN + c['betaNI']*lnI + c['betaNP']*lnP + c['betaNH']*lnH,
        'Irrigation':    c['betaI']  + c['betaII']*lnI + c['betaNI']*lnN + c['betaIP']*lnP + c['betaIH']*lnH,
        'Precipitation': c['betaP']  + c['betaPP']*lnP + c['betaNP']*lnN + c['betaIP']*lnI + c['betaPH']*lnH,
        'Herbicide':     c['betaH']  + c['betaHH']*lnH + c['betaNH']*lnN + c['betaIH']*lnI + c['betaPH']*lnP,
    }
    tot = sum(e.values())
    return {f: v/tot for f, v in e.items()}


def normalise_gamma(g: Dict[str, float], features: List[str]) -> Dict[str, float]:
    g   = {f: max(v, 0.0) for f, v in g.items()}
    tot = sum(g.values())
    if tot < 1e-12:
        return {f: 1.0/len(features) for f in features}
    return {f: v/tot for f, v in g.items()}


# Step e - jarque-bera non-gaussianity test
def jarque_bera_test(X: pd.DataFrame, features: List[str]) -> None:
    """
    Tests whether each feature's residuals (after regressing out other features)
    are non-Gaussian. LiNGAM requires non-Gaussian noise — if all features pass
    as Gaussian, fall back to DAGMA or GES.
    """
    print("\n  Jarque-Bera non-Gaussianity test (LiNGAM prerequisite):")
    print(f"  {'Feature':<30} {'JB stat':>10} {'p-value':>10} {'Non-Gaussian?':>14}")
    print(f"  {'-'*30} {'-'*10} {'-'*10} {'-'*14}")
    n_nongaussian = 0
    for f in features:
        others = [o for o in features if o != f]
        resid  = X[f].values - LinearRegression().fit(X[others].values, X[f].values).predict(X[others].values)
        jb, p  = jarque_bera(resid)
        ng     = "YES ok" if p < 0.05 else "no (Gaussian)"
        if p < 0.05:
            n_nongaussian += 1
        print(f"  {f:<30} {jb:>10.2f} {p:>10.4f} {ng:>14}")
    print(f"\n  {n_nongaussian}/{len(features)} features have non-Gaussian residuals")
    if n_nongaussian < len(features) // 2:
        print("  Warning: Majority Gaussian → LiNGAM assumptions may be weak → use DAGMA as primary")
    else:
        print("  ok Sufficient non-Gaussianity → LiNGAM is appropriate")


# Step a - directlingam graph recovery
def run_directlingam(X: pd.DataFrame, y: pd.Series,
                     features: List[str], target_name: str
                     ) -> Tuple[np.ndarray, Dict[str, float], nx.DiGraph]:
    """
    Runs DirectLiNGAM with target variable included as a node.
    Returns:
      B_matrix  : full adjacency matrix (n_vars × n_vars)
      lingam_gamma: gamma extracted from target row, normalised
      dag       : networkx DiGraph of recovered inter-feature edges
    """
    if not HAS_LINGAM:
        print("  x lingam not available, skipping")
        return None, {f: 1.0/len(features) for f in features}, nx.DiGraph()

    # Combine features + target into one DataFrame for LiNGAM
    all_cols  = features + [target_name]
    X_full    = X.copy()
    X_full[target_name] = y.values if hasattr(y, 'values') else y

    model = lingam.DirectLiNGAM()
    model.fit(X_full[all_cols].values)

    B = model.adjacency_matrix_   # shape: (n_vars, n_vars), B[i,j] = effect of j on i
    # B[i, j] represents the causal effect of variable j on variable i

    target_idx = all_cols.index(target_name)

    # Extract causal effects on target: B[target_idx, j] for j in features
    raw_effects = {}
    for j, f in enumerate(features):
        raw_effects[f] = abs(float(B[target_idx, j]))

    lingam_gamma = normalise_gamma(raw_effects, features)

    # Build DAG of inter-feature edges (exclude target)
    G = nx.DiGraph()
    G.add_nodes_from(features)
    feature_indices = {f: i for i, f in enumerate(all_cols)}
    threshold = 0.05  # minimum |β| to count as an edge

    for i, fi in enumerate(features):
        for j, fj in enumerate(features):
            if i != j and abs(B[i, j]) > threshold:
                G.add_edge(fj, fi, weight=float(B[i, j]))

    return B, lingam_gamma, G


def compare_graphs(recovered: nx.DiGraph, true_edges: set,
                   features: List[str]) -> dict:
    """
    Compare recovered inter-feature graph to true graph.
    Metrics: TP, FP, FN, precision, recall, F1, SHD.
    """
    recovered_edges = set(recovered.edges())
    tp  = len(recovered_edges & true_edges)
    fp  = len(recovered_edges - true_edges)
    fn  = len(true_edges - recovered_edges)
    shd = fp + fn  # Structural Hamming Distance (undirected)

    prec   = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1     = 2*prec*recall / (prec+recall) if (prec+recall) > 0 else 0.0

    return {
        'TP': tp, 'FP': fp, 'FN': fn,
        'Precision': round(prec, 3),
        'Recall':    round(recall, 3),
        'F1':        round(f1, 3),
        'SHD':       shd,
        'Recovered edges': list(recovered_edges),
        'True edges':      list(true_edges),
        'Missing edges':   list(true_edges - recovered_edges),
        'Extra edges':     list(recovered_edges - true_edges),
    }


# Step f - dagma linear (fallback)
def run_dagma(X: pd.DataFrame, y: pd.Series,
              features: List[str], target_name: str
              ) -> Tuple[np.ndarray, Dict[str, float], nx.DiGraph]:
    if not HAS_DAGMA:
        print("  x dagma not available, skipping")
        return None, {f: 1.0/len(features) for f in features}, nx.DiGraph()

    all_cols  = features + [target_name]
    X_full    = X.copy()
    X_full[target_name] = y.values if hasattr(y, 'values') else y
    data      = X_full[all_cols].values.astype(float)

    model = DagmaLinear(loss_type='l2')
    W     = model.fit(data, lambda1=0.02)  # W[i,j] = effect of i on j

    target_idx = all_cols.index(target_name)
    raw_effects = {}
    for j, f in enumerate(features):
        raw_effects[f] = abs(float(W[j, target_idx]))

    dagma_gamma = normalise_gamma(raw_effects, features)

    G = nx.DiGraph()
    G.add_nodes_from(features)
    threshold = 0.05
    for i, fi in enumerate(features):
        for j, fj in enumerate(features):
            if i != j and abs(W[i, j]) > threshold:
                G.add_edge(fi, fj, weight=float(W[i, j]))

    return W, dagma_gamma, G


class LinearMechanisms:
    def __init__(self, dag: nx.DiGraph, features: List[str]):
        self.dag = dag; self.features = features
        self.mech: Dict[str,dict] = {}
        self.topo: List[str] = list(nx.topological_sort(dag))

    def fit(self, X: pd.DataFrame) -> 'LinearMechanisms':
        for var in self.features:
            parents = list(self.dag.predecessors(var))
            if not parents:
                self.mech[var] = {'parents':[],'intercept':float(X[var].mean()),
                                  'coef':None,'sigma':float(X[var].std(ddof=0))}
            else:
                lr    = LinearRegression().fit(X[parents].values, X[var].values)
                resid = X[var].values - lr.predict(X[parents].values)
                self.mech[var] = {'parents':parents,'intercept':float(lr.intercept_),
                                  'coef':lr.coef_.astype(float),'sigma':float(np.std(resid,ddof=0))}
        return self

    def sample_batch(self, S: Set[str], x: pd.Series, M: int) -> np.ndarray:
        X_out = np.empty((M, len(self.features)))
        fi    = {f: i for i, f in enumerate(self.features)}
        sim   = {}
        for var in self.topo:
            vi = fi[var]
            if var in S:
                val = float(x[var]); X_out[:,vi] = val; sim[var] = np.full(M,val); continue
            m = self.mech[var]
            if not m['parents']:
                mu = np.full(M, m['intercept'])
            else:
                pm = np.stack([sim[p] if p in sim else np.full(M,float(x[p])) for p in m['parents']], axis=1)
                mu = m['intercept'] + pm @ m['coef']
            X_out[:,vi] = mu if m['sigma'] < 1e-8 else rng.normal(mu, m['sigma'], size=M)
            sim[var] = X_out[:,vi]
        return X_out


class PermutationEngine:
    """For yield (4 features, 500 perms)."""
    def __init__(self, model, sem, gamma, X_train):
        self.model = model; self.features = list(X_train.columns)
        self.gamma = gamma; self.sem = sem
        self._Ef  = float(model.predict(X_train.values).mean())

    def _vc(self, S, x, M):
        return float(np.mean(self.model.predict(self.sem.sample_batch(S, x, M))))

    def explain_instance(self, x, n_perms, M):
        fx  = float(self.model.predict(x.values.reshape(1,-1))[0])
        phi = np.zeros(len(self.features))
        for _ in range(n_perms):
            perm = rng.permutation(self.features); S = set(); v_prev = self._vc(S, x, M)
            for feat in perm:
                Sn = S|{feat}; vc = self._vc(Sn, x, M)
                phi[self.features.index(feat)] += self.gamma[feat]*(vc - v_prev)
                v_prev, S = vc, Sn
        phi /= n_perms
        s = float(phi.sum())
        if abs(s) > 1e-12: phi = phi * ((fx - self._Ef) / s)
        return pd.Series(phi, index=self.features)

    def explain_global(self, X_test, k, n_perms, M):
        k   = min(k, len(X_test))
        idx = rng.choice(len(X_test), size=k, replace=False)
        return pd.concat([self.explain_instance(X_test.iloc[i], n_perms, M) for i in idx], axis=1).abs().mean(axis=1)


class ExactEngine:
    """For soil (7 features, 2^7=128 subsets)."""
    def __init__(self, model, sem, gamma, X_train):
        self.model = model; self.features = list(X_train.columns)
        self.n = len(self.features); self.gamma = gamma; self.sem = sem
        self._Ef = float(model.predict(X_train.values).mean())
        self._kw = {sz: float(math.factorial(sz)*math.factorial(self.n-sz-1)/math.factorial(self.n))
                    for sz in range(self.n)}

    def explain_instance(self, x, M):
        fx  = float(self.model.predict(x.values.reshape(1,-1))[0])
        phi = np.zeros(self.n)
        cache = {frozenset(sub): float(np.mean(self.model.predict(self.sem.sample_batch(frozenset(sub), x, M))))
                 for r in range(self.n+1) for sub in itertools.combinations(self.features, r)}
        for idx, feat in enumerate(self.features):
            rest = [f for f in self.features if f != feat]
            for r in range(self.n):
                for sub in itertools.combinations(rest, r):
                    S = frozenset(sub)
                    phi[idx] += self._kw[len(sub)] * self.gamma[feat] * (cache[S|{feat}] - cache[S])
        s = float(phi.sum())
        if abs(s) > 1e-12: phi = phi * ((fx - self._Ef) / s)
        return pd.Series(phi, index=self.features)

    def explain_global(self, X_test, k, M):
        k = min(k, len(X_test))
        phi_list = []
        for i in range(k):
            if (i+1) % 20 == 0: print(f"      instance {i+1}/{k} ...", flush=True)
            phi_list.append(self.explain_instance(X_test.iloc[i], M))
        return pd.concat(phi_list, axis=1).abs().mean(axis=1)


# Metrics
def compute_metrics(true_gamma: Dict[str,float], global_abs: pd.Series,
                    features: List[str]) -> dict:
    gt   = np.array([true_gamma[f]        for f in features])
    raw  = np.array([global_abs.get(f,0)  for f in features])
    sv   = raw/raw.sum() if raw.sum() > 1e-12 else raw
    pr,_ = pearsonr(sv, gt); sr,_ = spearmanr(sv, gt)
    cos  = float(np.dot(sv,gt)/(np.linalg.norm(sv)*np.linalg.norm(gt)+1e-12))
    rmse = float(np.sqrt(np.mean((sv-gt)**2)))
    mae  = float(np.mean(np.abs(sv-gt)))
    return {'Pearson_r':round(float(pr),4),'Spearman_r':round(float(sr),4),
            'Cosine':round(cos,4),'RMSE':round(rmse,4),'MAE':round(mae,4),
            'shap_normalized': {f: round(sv[i],4) for i,f in enumerate(features)}}


def gamma_metrics(true_gamma: Dict[str,float], est_gamma: Dict[str,float],
                  features: List[str]) -> dict:
    """Compare estimated gamma directly to true gamma."""
    gt = np.array([true_gamma[f] for f in features])
    eg = np.array([est_gamma[f]  for f in features])
    pr,_ = pearsonr(eg, gt); sr,_ = spearmanr(eg, gt)
    rmse = float(np.sqrt(np.mean((eg-gt)**2)))
    mae  = float(np.mean(np.abs(eg-gt)))
    return {'Gamma_Pearson':round(float(pr),4),'Gamma_Spearman':round(float(sr),4),
            'Gamma_RMSE':round(rmse,4),'Gamma_MAE':round(mae,4)}


# Main runner for one dataset
def run_dataset(name: str, features: List[str], target_name: str,
                true_edges: set, true_gamma: Dict[str,float],
                Xtr: pd.DataFrame, Xte: pd.DataFrame,
                ytr: np.ndarray, yte: np.ndarray,
                engine_type: str, cfg: dict) -> pd.DataFrame:

    print(f"\n{'='*80}")
    print(f"DATASET: {name.upper()}")
    print(f"{'='*80}")
    print(f"\n  True gamma: { {f: round(v,4) for f,v in true_gamma.items()} }")
    print(f"  True inter-feature edges: {true_edges if true_edges else 'none (star DAG)'}")

    # Train XGBoost (once, shared)
    model = XGBRegressor(n_estimators=160, max_depth=5, learning_rate=0.08,
                         subsample=0.9, colsample_bytree=0.9,
                         random_state=cfg['RANDOM_SEED'], verbosity=0,
                         n_jobs=1, tree_method='hist')
    model.fit(Xtr.values, ytr)

    # ── E: Jarque-Bera test ──────────────────────────────────────
    print("\n--- STEP E: Non-Gaussianity check ---")
    jarque_bera_test(Xtr, features)

    results = []

    # ── A: DirectLiNGAM ─────────────────────────────────────────
    print("\n--- STEP A: DirectLiNGAM ---")
    t0 = time.time()
    B_lingam, lingam_gamma, lingam_dag = run_directlingam(Xtr, ytr, features, target_name)
    t_lingam = time.time() - t0

    if B_lingam is not None:
        graph_cmp = compare_graphs(lingam_dag, true_edges, features)
        gm        = gamma_metrics(true_gamma, lingam_gamma, features)

        print(f"\n  LiNGAM recovered γ: { {f: round(v,4) for f,v in lingam_gamma.items()} }")
        print(f"  Gamma Pearson vs GT: {gm['Gamma_Pearson']:.4f}")
        print(f"\n  Graph comparison:")
        print(f"    TP={graph_cmp['TP']}  FP={graph_cmp['FP']}  FN={graph_cmp['FN']}")
        print(f"    Precision={graph_cmp['Precision']:.3f}  Recall={graph_cmp['Recall']:.3f}  F1={graph_cmp['F1']:.3f}  SHD={graph_cmp['SHD']}")
        print(f"    Recovered edges: {graph_cmp['Recovered edges']}")
        print(f"    Missing edges:   {graph_cmp['Missing edges']}")
        print(f"    Extra edges:     {graph_cmp['Extra edges']}")

        # Run Causal SHAP with LiNGAM gamma + LiNGAM graph
        print(f"\n  Running Causal SHAP with LiNGAM γ + LiNGAM graph...")
        sem_lingam = LinearMechanisms(lingam_dag, features).fit(Xtr)
        if engine_type == 'permutation':
            engine = PermutationEngine(model, sem_lingam, lingam_gamma, Xtr)
            g_abs  = engine.explain_global(Xte, cfg['YIELD_K'], cfg['YIELD_MAX_PERM'], cfg['YIELD_M'])
        else:
            engine = ExactEngine(model, sem_lingam, lingam_gamma, Xtr)
            g_abs  = engine.explain_global(Xte, cfg['SOIL_K'], cfg['SOIL_M'])

        m = compute_metrics(true_gamma, g_abs, features)
        print(f"  Attribution Pearson={m['Pearson_r']:.4f} | Spearman={m['Spearman_r']:.4f} | "
              f"Cosine={m['Cosine']:.4f} | RMSE={m['RMSE']:.4f}")

        results.append({
            'Method': 'LiNGAM (graph + gamma)',
            'Graph_TP': graph_cmp['TP'], 'Graph_FP': graph_cmp['FP'],
            'Graph_FN': graph_cmp['FN'], 'Graph_SHD': graph_cmp['SHD'],
            'Graph_F1': graph_cmp['F1'],
            **gm, **m, 'Time_s': round(t_lingam, 1),
        })

        # Also: LiNGAM gamma + TRUE graph
        print(f"\n  Running Causal SHAP with LiNGAM γ + TRUE graph...")
        true_dag = nx.DiGraph(); true_dag.add_nodes_from(features)
        for u,v in true_edges: true_dag.add_edge(u,v)
        sem_true  = LinearMechanisms(true_dag, features).fit(Xtr)
        if engine_type == 'permutation':
            engine2 = PermutationEngine(model, sem_true, lingam_gamma, Xtr)
            g_abs2  = engine2.explain_global(Xte, cfg['YIELD_K'], cfg['YIELD_MAX_PERM'], cfg['YIELD_M'])
        else:
            engine2 = ExactEngine(model, sem_true, lingam_gamma, Xtr)
            g_abs2  = engine2.explain_global(Xte, cfg['SOIL_K'], cfg['SOIL_M'])
        m2 = compute_metrics(true_gamma, g_abs2, features)
        print(f"  Attribution Pearson={m2['Pearson_r']:.4f} | Spearman={m2['Spearman_r']:.4f} | "
              f"Cosine={m2['Cosine']:.4f} | RMSE={m2['RMSE']:.4f}")
        results.append({
            'Method': 'LiNGAM gamma + True graph',
            'Graph_TP': len(true_edges),'Graph_FP':0,'Graph_FN':0,
            'Graph_SHD':0,'Graph_F1':1.0,
            **gm, **m2, 'Time_s': round(t_lingam, 1),
        })

    # ── f: dagma ─────────────────────────────────────────────────
    print("\n--- STEP F: DAGMA (linear) ---")
    t0 = time.time()
    W_dagma, dagma_gamma, dagma_dag = run_dagma(Xtr, ytr, features, target_name)
    t_dagma = time.time() - t0

    if W_dagma is not None:
        graph_cmp_d = compare_graphs(dagma_dag, true_edges, features)
        gm_d        = gamma_metrics(true_gamma, dagma_gamma, features)

        print(f"\n  DAGMA recovered γ: { {f: round(v,4) for f,v in dagma_gamma.items()} }")
        print(f"  Gamma Pearson vs GT: {gm_d['Gamma_Pearson']:.4f}")
        print(f"  Graph: TP={graph_cmp_d['TP']} FP={graph_cmp_d['FP']} FN={graph_cmp_d['FN']} SHD={graph_cmp_d['SHD']}")

        sem_dagma = LinearMechanisms(dagma_dag, features).fit(Xtr)
        if engine_type == 'permutation':
            engine3 = PermutationEngine(model, sem_dagma, dagma_gamma, Xtr)
            g_abs3  = engine3.explain_global(Xte, cfg['YIELD_K'], cfg['YIELD_MAX_PERM'], cfg['YIELD_M'])
        else:
            engine3 = ExactEngine(model, sem_dagma, dagma_gamma, Xtr)
            g_abs3  = engine3.explain_global(Xte, cfg['SOIL_K'], cfg['SOIL_M'])

        m3 = compute_metrics(true_gamma, g_abs3, features)
        print(f"  Attribution Pearson={m3['Pearson_r']:.4f} | Spearman={m3['Spearman_r']:.4f} | "
              f"Cosine={m3['Cosine']:.4f} | RMSE={m3['RMSE']:.4f}")
        results.append({
            'Method': 'DAGMA (graph + gamma)',
            'Graph_TP': graph_cmp_d['TP'],'Graph_FP': graph_cmp_d['FP'],
            'Graph_FN': graph_cmp_d['FN'],'Graph_SHD': graph_cmp_d['SHD'],
            'Graph_F1': graph_cmp_d['F1'],
            **gm_d, **m3, 'Time_s': round(t_dagma, 1),
        })

    # ── Baselines: true gamma + true graph ───────────────────────
    print("\n--- Baseline: True gamma + True graph ---")
    true_dag = nx.DiGraph(); true_dag.add_nodes_from(features)
    for u,v in true_edges: true_dag.add_edge(u,v)
    sem_true = LinearMechanisms(true_dag, features).fit(Xtr)
    if engine_type == 'permutation':
        engine_b = PermutationEngine(model, sem_true, true_gamma, Xtr)
        g_abs_b  = engine_b.explain_global(Xte, cfg['YIELD_K'], cfg['YIELD_MAX_PERM'], cfg['YIELD_M'])
    else:
        engine_b = ExactEngine(model, sem_true, true_gamma, Xtr)
        g_abs_b  = engine_b.explain_global(Xte, cfg['SOIL_K'], cfg['SOIL_M'])
    mb = compute_metrics(true_gamma, g_abs_b, features)
    print(f"  Attribution Pearson={mb['Pearson_r']:.4f} | Spearman={mb['Spearman_r']:.4f} | "
          f"Cosine={mb['Cosine']:.4f} | RMSE={mb['RMSE']:.4f}")
    results.append({
        'Method': 'True gamma + True graph (ceiling)',
        'Graph_TP':len(true_edges),'Graph_FP':0,'Graph_FN':0,
        'Graph_SHD':0,'Graph_F1':1.0,
        'Gamma_Pearson':1.0,'Gamma_Spearman':1.0,'Gamma_RMSE':0.0,'Gamma_MAE':0.0,
        **mb, 'Time_s': 0.0,
    })

    df = pd.DataFrame(results)
    print(f"\n{'='*80}")
    print(f"RESULTS SUMMARY — {name.upper()}")
    print(f"{'='*80}")
    disp = ['Method','Gamma_Pearson','Graph_F1','Graph_SHD','Pearson_r','Spearman_r','RMSE']
    print(df[disp].to_string(index=False))
    return df


# Main
def main():
    print("\n" + "="*80)
    print("DIRECTLINGAM + DAGMA — CAUSAL GRAPH & GAMMA DISCOVERY")
    print("Soil and Yield Datasets")
    print("="*80)

    all_results = {}

    # Yield
    data        = np.load(CONFIG['YIELD_DATA_PATH'], allow_pickle=True)
    X_raw_train = pd.DataFrame(data['X_train'], columns=YIELD_FEATURES)
    X_raw_test  = pd.DataFrame(data['X_test'],  columns=YIELD_FEATURES)
    y_train_y   = data['y_train'].astype(float)
    y_test_y    = data['y_test'].astype(float)

    true_gamma_y = compute_yield_gamma(X_raw_train)

    scaler_y = StandardScaler()
    Xtr_y = pd.DataFrame(scaler_y.fit_transform(X_raw_train), columns=YIELD_FEATURES)
    Xte_y = pd.DataFrame(scaler_y.transform(X_raw_test),      columns=YIELD_FEATURES)

    df_y = run_dataset(
        name='Yield', features=YIELD_FEATURES, target_name=YIELD_TARGET,
        true_edges=YIELD_TRUE_EDGES, true_gamma=true_gamma_y,
        Xtr=Xtr_y, Xte=Xte_y, ytr=y_train_y, yte=y_test_y,
        engine_type='permutation', cfg=CONFIG,
    )
    df_y.to_csv('lingam_results_yield.csv', index=False)

    # Soil
    dc       = pd.read_csv(CONFIG['SOIL_DATA_PATH']).rename(columns=SOIL_COL_MAP)
    Xtr_s_raw, Xte_s_raw, ytr_s, yte_s = train_test_split(
        dc[SOIL_FEATURES], dc[SOIL_TARGET],
        test_size=CONFIG['TEST_SIZE'], random_state=CONFIG['RANDOM_SEED'], shuffle=True)

    true_gamma_s = compute_soil_gamma(CONFIG['SOIL_GROUND_TRUTH_PATH'])

    scaler_s = StandardScaler()
    Xtr_s = pd.DataFrame(scaler_s.fit_transform(Xtr_s_raw), columns=SOIL_FEATURES, index=Xtr_s_raw.index)
    Xte_s = pd.DataFrame(scaler_s.transform(Xte_s_raw),     columns=SOIL_FEATURES, index=Xte_s_raw.index)

    df_s = run_dataset(
        name='Soil', features=SOIL_FEATURES, target_name=SOIL_TARGET,
        true_edges=SOIL_TRUE_EDGES, true_gamma=true_gamma_s,
        Xtr=Xtr_s, Xte=Xte_s, ytr=ytr_s.values, yte=yte_s.values,
        engine_type='exact', cfg=CONFIG,
    )
    df_s.to_csv('lingam_results_soil.csv', index=False)

    print(f"\n\nSaved: lingam_results_yield.csv, lingam_results_soil.csv")


if __name__ == '__main__':
    main()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 145.0 kB/s  0:00:10 eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 146.0 kB/s  0:00:37 eta 0:00:02
  Created wheel for semopy: filename=semopy-2.3.11-py3-none-any.whl size=1659769 sha256=3055fd5736550fdca27cfae906c7f8e703a176662ca88401eb186678266bd0c0
  Stored in directory: .cache/pip/wheels/c6/24/8b/be911b059a61f490f38425eb19bf2fed470a5ead97228e8255
Successfully built semopy
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [lingam]10/11 [lingam]sbar2]

[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
✓ lingam available
✓ dagma available

DIRECTLINGAM + DAGMA — CAUSAL GRAPH & GAMMA DISCOVERY
Soil and Yield Datasets

DATASET: YIELD

  True gamma: {'Nitrogen': 0.3118, 'Irrigation': 0.3727, 'Precipitation': 0.2457, 'Her


  DAGMA recovered γ: {'Nitrogen': 0.2421, 'Irrigation': 0.3778, 'Precipitation': 0.2727, 'Herbicide': 0.1074}
  Gamma Pearson vs GT: 0.9326
  Graph: TP=0 FP=3 FN=0 SHD=3
  Attribution Pearson=0.9116 | Spearman=0.8000 | Cosine=0.9651 | RMSE=0.0809

--- Baseline: True gamma + True graph ---
  Attribution Pearson=0.8651 | Spearman=0.8000 | Cosine=0.9325 | RMSE=0.1191

RESULTS SUMMARY — YIELD
                           Method  Gamma_Pearson  Graph_F1  Graph_SHD  Pearson_r  Spearman_r   RMSE
           LiNGAM (graph + gamma)            NaN       0.0          6     0.8527         0.8 0.0865
        LiNGAM gamma + True graph            NaN       1.0          0     0.9055         0.8 0.0682
            DAGMA (graph + gamma)         0.9326       0.0          3     0.9116         0.8 0.0809
True gamma + True graph (ceiling)         1.0000       1.0          0     0.8651         0.8 0.1191

DATASET: SOIL

  True gamma: {'Organic Carbon': 0.0622, 'Nitrogen': 0.4012, 'Phosphorus': 0.0762, 'Potassi


  DAGMA recovered γ: {'Organic Carbon': 0.0578, 'Nitrogen': 0.3589, 'Phosphorus': 0.0517, 'Potassium': 0.2169, 'pH': 0.178, 'Zinc': 0.0236, 'Electrical Conductivity': 0.1131}
  Gamma Pearson vs GT: 0.9466
  Graph: TP=1 FP=0 FN=1 SHD=1
      instance 20/100 ...
      instance 40/100 ...
      instance 60/100 ...
      instance 80/100 ...
      instance 100/100 ...
  Attribution Pearson=0.9869 | Spearman=0.9286 | Cosine=0.9821 | RMSE=0.0504

--- Baseline: True gamma + True graph ---
      instance 20/100 ...
      instance 40/100 ...
      instance 60/100 ...
      instance 80/100 ...
      instance 100/100 ...
  Attribution Pearson=0.9869 | Spearman=0.8214 | Cosine=0.9751 | RMSE=0.0642

RESULTS SUMMARY — SOIL
                           Method  Gamma_Pearson  Graph_F1  Graph_SHD  Pearson_r  Spearman_r   RMSE
           LiNGAM (graph + gamma)        -0.0311     0.133         13     0.1670      0.1786 0.1884
        LiNGAM gamma + True graph        -0.0311     1.000          0     0.1716 

In [1]:
#!/usr/bin/env python3
"""
COMPAS Classification: DAGMA / DAGMA-DCE / PC -> Causal SHAP vs MC Shapley
============================================================================

Purpose
-------
Real-world, no-ground-truth attribution comparison on COMPAS.

This script compares:
  1. Standard DAGMA -> graph/gamma-guided Causal SHAP
  2. DAGMA-DCE MLP -> graph/gamma-guided Causal SHAP
  3. PC Algorithm -> graph/gamma-guided Causal SHAP
  4. Monte Carlo Shapley reference baseline

Metrics computed exactly as requested:
  - Faithfulness Spearman ↑
  - Predictive Alignment Pearson ↑  [vs MC Shapley reference]
  - Attribution Cosine ↑            [vs MC Shapley reference]
  - Robustness Cosine ↑
  - Rank Robustness ↑
  - L2 ↓

Important thesis claim boundary
-------------------------------
COMPAS has no ground-truth causal graph or ground-truth feature attribution.
Therefore, this experiment should be presented as:

  "A no-ground-truth real-world comparison of graph-guided causal-SHAP-style
   methods against Monte Carlo Shapley as a model-agnostic reference."

Do NOT claim that DAGMA/DAGMA-DCE/PC recover the true causal graph.
Do NOT claim verified causal correctness.

Preprocessing fix
-----------------
The provided COMPAS NPZ contains complementary one-hot columns such as:
  sex_0 / sex_1, c_charge_degree_F / c_charge_degree_M,
  race_African-American_0 / race_African-American_1, etc.

For DAG discovery, complementary columns create perfect anti-correlations and
unstable edges. This script keeps a reduced, non-redundant feature set:
  - numeric columns are kept
  - one binary indicator per binary variable is kept
  - one reference category is dropped for multicategory groups
"""

import sys
sys.path.insert(0, "DAGMA-DCE/src")

import time
import warnings
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import networkx as nx

from scipy.stats import pearsonr, spearmanr
from sklearn.linear_model import LinearRegression
from xgboost import XGBClassifier

from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.cit import fisherz

warnings.filterwarnings("ignore")

try:
    from dagma.linear import DagmaLinear
    HAS_DAGMA = True
except Exception as e:
    HAS_DAGMA = False
    print(f"WARNING: dagma not available: {e}")

try:
    from DagmaDCE import nonlinear_dce
    HAS_DCE = True
except Exception as e:
    HAS_DCE = False
    print(f"WARNING: DagmaDCE not available: {e}")


# Configuration

@dataclass
class Config:
    data_path: str = "../../../datasets/compas_preprocessed.npz"
    random_seed: int = 42

    # Model
    n_estimators: int = 250
    max_depth: int = 4
    learning_rate: float = 0.05

    # Causal SHAP settings
    causal_k: int = 40
    causal_m: int = 64
    causal_perms: int = 200

    # MC Shapley reference settings
    # This is the most expensive part.
    mc_k: int = 40
    mc_bg_size: int = 120
    mc_iterations: int = 600

    # Metrics
    faith_k: int = 40
    robust_k: int = 15
    perturb_sigma: float = 0.05
    perturb_runs: int = 5

    # Graph thresholds
    dagma_graph_threshold: float = 0.05
    dce_graph_threshold: float = 0.10

    # Dagma
    dagma_lambda1_graph: float = 0.03
    dagma_lambda1_gamma: float = 0.02

    # Dagma-dce
    dce_lambda1_graph: float = 5e-2
    dce_lambda1_gamma: float = 3.5e-2
    dce_lambda2: float = 5e-3
    dce_lr: float = 2e-4
    dce_mu_factor: float = 0.1
    dce_mu_init: float = 0.1
    dce_T: int = 4
    dce_warm_iter: int = 7000
    dce_max_iter: int = 8000
    dce_hidden_mult: int = 3

    # PC
    pc_alpha: float = 0.05


CFG = Config()


# Utilities

def set_global_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)


def normalize_vec(values: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    values = np.maximum(np.asarray(values, dtype=float), 0.0)
    s = values.sum()
    if s <= eps:
        return np.ones_like(values, dtype=float) / len(values)
    return values / s


def normalize_signed_abs(values: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    values = np.abs(np.asarray(values, dtype=float))
    s = values.sum()
    if s <= eps:
        return np.ones_like(values, dtype=float) / len(values)
    return values / s


def normalize_gamma_dict(effects: Dict[str, float], features: List[str]) -> Dict[str, float]:
    arr = normalize_vec(np.array([effects.get(f, 0.0) for f in features], dtype=float))
    return {f: float(arr[i]) for i, f in enumerate(features)}


def safe_spearman(a: np.ndarray, b: np.ndarray) -> float:
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    if np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return np.nan
    r, _ = spearmanr(a, b)
    return float(r) if not np.isnan(r) else np.nan


def safe_pearson(a: np.ndarray, b: np.ndarray) -> float:
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    if np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return np.nan
    r, _ = pearsonr(a, b)
    return float(r) if not np.isnan(r) else np.nan


def cosine(a: np.ndarray, b: np.ndarray) -> float:
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))


def make_dag_greedy(G: nx.DiGraph) -> nx.DiGraph:
    """
    Convert a directed graph to a DAG by keeping edges greedily only if they
    do not create a directed cycle.

    This is a practical step for Causal SHAP sampling. It should not be
    interpreted as a validated causal orientation rule.
    """
    dag = nx.DiGraph()
    dag.add_nodes_from(G.nodes())
    for u, v, data in G.edges(data=True):
        dag.add_edge(u, v, **data)
        if not nx.is_directed_acyclic_graph(dag):
            dag.remove_edge(u, v)
    return dag


def build_graph_from_W(W: np.ndarray, features: List[str], threshold: float) -> nx.DiGraph:
    G = nx.DiGraph()
    G.add_nodes_from(features)
    d = len(features)
    for i, fi in enumerate(features):
        for j, fj in enumerate(features):
            if i != j and abs(float(W[i, j])) > threshold:
                G.add_edge(fi, fj, weight=float(W[i, j]))
    return make_dag_greedy(G)


# COMPAS loading and preprocessing

def load_compas(cfg: Config) -> Tuple[pd.DataFrame, pd.DataFrame, np.ndarray, np.ndarray, List[str]]:
    raw = np.load(cfg.data_path, allow_pickle=True)
    feature_names = [str(f) for f in raw["feature_names"]]

    X_train_full = pd.DataFrame(raw["X_train"], columns=feature_names)
    X_test_full = pd.DataFrame(raw["X_test"], columns=feature_names)
    y_train = raw["y_train"].astype(int)
    y_test = raw["y_test"].astype(int)

    X_train, X_test, kept_features, dropped_features = preprocess_compas_features(
        X_train_full, X_test_full
    )

    print("\nPreprocessing summary")
    print("---------------------")
    print(f"Original features: {len(feature_names)}")
    print(f"Kept features:     {len(kept_features)}")
    print(f"Dropped features:  {len(dropped_features)}")
    print(f"Kept:   {kept_features}")
    print(f"Dropped:{dropped_features}")

    return X_train, X_test, y_train, y_test, kept_features


def preprocess_compas_features(
    X_train_full: pd.DataFrame,
    X_test_full: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame, List[str], List[str]]:
    """
    COMPAS-specific redundancy removal.

    Keeps:
      - age, juvenile counts, priors_count
      - c_charge_degree_F, with M as reference
      - sex_1, with sex_0 as reference
      - age_cat_25-45_1 and age_cat_Greaterthan45_1,
        with age_cat_Lessthan25_1 as reference
      - race_African-American_1, with Caucasian as reference

    Drops:
      - complementary _0 columns
      - redundant binary complements
      - one category from each one-hot group as reference
    """

    cols = list(X_train_full.columns)

    preferred_order = [
        "age",
        "juv_fel_count",
        "juv_misd_count",
        "juv_other_count",
        "priors_count",

        # Binary charge degree: keep F, drop M.
        "c_charge_degree_F",

        # Binary sex indicator: keep sex_1, drop sex_0.
        "sex_1",

        # Age categories: keep two dummies, drop <25 as reference.
        "age_cat_25-45_1",
        "age_cat_Greaterthan45_1",

        # Race: keep one dummy, use Caucasian/other as reference.
        "race_African-American_1",
    ]

    kept = [c for c in preferred_order if c in cols]

    # Fallback: if user has a slightly different COMPAS preprocessing, keep:
    # numeric columns + columns ending in _1, but avoid keeping every category.
    if len(kept) < 6:
        numeric_like = [
            c for c in cols
            if not (c.endswith("_0") or c.endswith("_1"))
        ]
        one_indicators = [c for c in cols if c.endswith("_1")]
        kept = numeric_like + one_indicators

        # Drop obvious reference categories if present.
        for ref in ["age_cat_Lessthan25_1", "race_Caucasian_1"]:
            if ref in kept:
                kept.remove(ref)

        # Drop one of binary complements if both still slipped in.
        for ref in ["c_charge_degree_M", "sex_0"]:
            if ref in kept:
                kept.remove(ref)

    # Remove zero-variance columns after selection.
    kept_nonconstant = []
    for c in kept:
        if c in X_train_full.columns and float(X_train_full[c].std(ddof=0)) > 1e-12:
            kept_nonconstant.append(c)

    kept = kept_nonconstant
    dropped = [c for c in cols if c not in kept]

    return (
        X_train_full[kept].copy(),
        X_test_full[kept].copy(),
        kept,
        dropped,
    )


def fit_xgb_binary(X_train: pd.DataFrame, y_train: np.ndarray, cfg: Config) -> XGBClassifier:
    model = XGBClassifier(
        n_estimators=cfg.n_estimators,
        max_depth=cfg.max_depth,
        learning_rate=cfg.learning_rate,
        subsample=0.85,
        colsample_bytree=0.85,
        random_state=cfg.random_seed,
        verbosity=0,
        n_jobs=1,
        tree_method="hist",
        objective="binary:logistic",
        eval_metric="logloss",
    )
    model.fit(X_train.values, y_train)
    return model


# Dagma

def run_dagma_graph(X_train: pd.DataFrame, features: List[str], cfg: Config) -> nx.DiGraph:
    """
    Learn feature-feature DAG only.

    Probability/model-output nodes are intentionally NOT included here because
    they can distort the feature graph.
    """
    if not HAS_DAGMA:
        print("  DAGMA unavailable. Returning empty graph.")
        G = nx.DiGraph()
        G.add_nodes_from(features)
        return G

    W = DagmaLinear(loss_type="l2").fit(
        X_train[features].values.astype(float),
        lambda1=cfg.dagma_lambda1_graph,
    )
    return build_graph_from_W(W, features, cfg.dagma_graph_threshold)


def run_dagma_gamma(
    X_train: pd.DataFrame,
    model: XGBClassifier,
    features: List[str],
    cfg: Config,
) -> Dict[int, Dict[str, float]]:
    """
    Class-specific feature -> model probability sensitivity using linear DAGMA.

    gamma_by_class[1] is based on edges to prob_cls1.
    gamma_by_class[0] is based on edges to prob_cls0 = 1 - prob_cls1.

    This is model-output sensitivity, not a ground-truth causal effect.
    """
    if not HAS_DAGMA:
        return {c: {f: 1.0 / len(features) for f in features} for c in [0, 1]}

    probs_pos = model.predict_proba(X_train[features].values)[:, 1]
    Xfull = X_train[features].copy()
    Xfull["prob_cls0"] = 1.0 - probs_pos
    Xfull["prob_cls1"] = probs_pos

    all_cols = features + ["prob_cls0", "prob_cls1"]
    W = DagmaLinear(loss_type="l2").fit(
        Xfull[all_cols].values.astype(float),
        lambda1=cfg.dagma_lambda1_gamma,
    )

    gamma_by_class = {}
    for c in [0, 1]:
        t = all_cols.index(f"prob_cls{c}")
        effects = {f: abs(float(W[i, t])) for i, f in enumerate(features)}
        gamma_by_class[c] = normalize_gamma_dict(effects, features)

    return gamma_by_class


# Dagma-dce

def _fit_dce_matrix(data_np: np.ndarray, cfg: Config, lambda1: float) -> np.ndarray:
    if not HAS_DCE:
        raise RuntimeError("DagmaDCE package is not available.")

    n_vars = data_np.shape[1]
    hidden = max(10, cfg.dce_hidden_mult * n_vars)

    data = torch.tensor(data_np.astype(np.float64), dtype=torch.float64)
    eq_model = nonlinear_dce.DagmaMLP_DCE(dims=[n_vars, hidden, 1], bias=True)
    model = nonlinear_dce.DagmaDCE(eq_model, use_mse_loss=True)

    W_tensor = model.fit(
        data,
        lambda1=lambda1,
        lambda2=cfg.dce_lambda2,
        lr=cfg.dce_lr,
        mu_factor=cfg.dce_mu_factor,
        mu_init=cfg.dce_mu_init,
        T=cfg.dce_T,
        warm_iter=cfg.dce_warm_iter,
        max_iter=cfg.dce_max_iter,
    )

    if isinstance(W_tensor, torch.Tensor):
        return W_tensor.detach().cpu().numpy()
    return np.asarray(W_tensor)


def run_dce_graph(X_train: pd.DataFrame, features: List[str], cfg: Config) -> nx.DiGraph:
    """
    Learn feature-feature graph only using DAGMA-DCE.
    """
    if not HAS_DCE:
        print("  DAGMA-DCE unavailable. Returning empty graph.")
        G = nx.DiGraph()
        G.add_nodes_from(features)
        return G

    print("  Fitting DAGMA-DCE graph on features only...")
    W = _fit_dce_matrix(X_train[features].values, cfg, cfg.dce_lambda1_graph)
    return build_graph_from_W(W, features, cfg.dce_graph_threshold)


def run_dce_gamma(
    X_train: pd.DataFrame,
    model: XGBClassifier,
    features: List[str],
    cfg: Config,
) -> Dict[int, Dict[str, float]]:
    """
    Class-specific feature -> model probability DCE.
    """
    if not HAS_DCE:
        return {c: {f: 1.0 / len(features) for f in features} for c in [0, 1]}

    probs_pos = model.predict_proba(X_train[features].values)[:, 1]
    Xfull = X_train[features].copy()
    Xfull["prob_cls0"] = 1.0 - probs_pos
    Xfull["prob_cls1"] = probs_pos

    all_cols = features + ["prob_cls0", "prob_cls1"]
    print("  Fitting DAGMA-DCE gamma on features + probability nodes...")
    W = _fit_dce_matrix(Xfull[all_cols].values, cfg, cfg.dce_lambda1_gamma)

    gamma_by_class = {}
    for c in [0, 1]:
        t = all_cols.index(f"prob_cls{c}")
        effects = {f: abs(float(W[i, t])) for i, f in enumerate(features)}
        gamma_by_class[c] = normalize_gamma_dict(effects, features)

    return gamma_by_class


# PC Algorithm

def run_pc_graph(X_train: pd.DataFrame, features: List[str], cfg: Config) -> nx.DiGraph:
    """
    Learn feature-feature graph with PC.

    Since PC may output partially directed/undirected structures, this function
    orients uncertain adjacencies greedily for Causal SHAP sampling. Do not
    overclaim these orientations as true causal directions.
    """
    cg = pc(
        X_train[features].values.astype(float),
        alpha=cfg.pc_alpha,
        indep_test=fisherz,
        node_names=features,
        verbose=False,
        show_progress=False,
    )

    adj = cg.G.graph
    nodes = [n.get_name() for n in cg.G.get_nodes()]

    G = nx.DiGraph()
    G.add_nodes_from(features)

    d = len(features)
    for i in range(d):
        for j in range(i + 1, d):
            if adj[i, j] != 0 or adj[j, i] != 0:
                # Common causallearn encoding for directed endpoint.
                if adj[i, j] == 1 and adj[j, i] == -1:
                    G.add_edge(nodes[i], nodes[j])
                elif adj[i, j] == -1 and adj[j, i] == 1:
                    G.add_edge(nodes[j], nodes[i])
                else:
                    # Uncertain/undirected: orient by column order to get a DAG.
                    G.add_edge(nodes[i], nodes[j])

    return make_dag_greedy(G)


def run_pc_gamma(
    X_train: pd.DataFrame,
    model: XGBClassifier,
    features: List[str],
    cfg: Config,
) -> Dict[int, Dict[str, float]]:
    """
    PC has no natural edge magnitude.

    For each class probability node, features adjacent to that probability node
    receive equal mass. This is intentionally conservative and should be reported
    as PC-connectivity gamma, not effect-magnitude gamma.
    """
    probs_pos = model.predict_proba(X_train[features].values)[:, 1]
    gamma_by_class = {}

    for c in [0, 1]:
        target = f"prob_cls{c}"
        Xfull = X_train[features].copy()
        Xfull[target] = probs_pos if c == 1 else 1.0 - probs_pos
        all_cols = features + [target]

        cg = pc(
            Xfull[all_cols].values.astype(float),
            alpha=cfg.pc_alpha,
            indep_test=fisherz,
            node_names=all_cols,
            verbose=False,
            show_progress=False,
        )

        adj = cg.G.graph
        nodes = [n.get_name() for n in cg.G.get_nodes()]
        t = nodes.index(target)

        connected = []
        for f in features:
            i = nodes.index(f)
            if adj[i, t] != 0 or adj[t, i] != 0:
                connected.append(f)

        if connected:
            gamma_by_class[c] = {
                f: (1.0 / len(connected) if f in connected else 0.0)
                for f in features
            }
        else:
            gamma_by_class[c] = {f: 1.0 / len(features) for f in features}

    return gamma_by_class


# Linear Gaussian SEM and Causal SHAP

class LinearGaussianSEM:
    def __init__(self, dag: nx.DiGraph, features: List[str]):
        self.dag = make_dag_greedy(dag)
        self.features = features
        self.topo = list(nx.topological_sort(self.dag))
        self.mech = {}

    def fit(self, X: pd.DataFrame):
        for var in self.features:
            parents = list(self.dag.predecessors(var))
            if not parents:
                self.mech[var] = {
                    "parents": [],
                    "intercept": float(X[var].mean()),
                    "coef": None,
                    "sigma": float(X[var].std(ddof=0)),
                }
            else:
                lr = LinearRegression().fit(X[parents].values, X[var].values)
                pred = lr.predict(X[parents].values)
                resid = X[var].values - pred
                self.mech[var] = {
                    "parents": parents,
                    "intercept": float(lr.intercept_),
                    "coef": lr.coef_.astype(float),
                    "sigma": float(np.std(resid, ddof=0)),
                }
        return self

    def sample_batch(self, S: set, x: pd.Series, M: int, rng: np.random.Generator) -> np.ndarray:
        X_out = np.empty((M, len(self.features)), dtype=float)
        idx = {f: k for k, f in enumerate(self.features)}
        sim = {}

        for var in self.topo:
            vi = idx[var]
            if var in S:
                val = float(x[var])
                X_out[:, vi] = val
                sim[var] = np.full(M, val)
                continue

            m = self.mech[var]
            if not m["parents"]:
                mu = np.full(M, m["intercept"])
            else:
                parent_mat = np.stack(
                    [sim[p] if p in sim else np.full(M, float(x[p])) for p in m["parents"]],
                    axis=1,
                )
                mu = m["intercept"] + parent_mat @ m["coef"]

            sigma = m["sigma"]
            X_out[:, vi] = mu if sigma < 1e-10 else rng.normal(mu, sigma, size=M)
            sim[var] = X_out[:, vi]

        return X_out


class CausalShapExplainer:
    def __init__(
        self,
        model: XGBClassifier,
        X_train: pd.DataFrame,
        dag: nx.DiGraph,
        gamma_by_class: Dict[int, Dict[str, float]],
        features: List[str],
        cfg: Config,
    ):
        self.model = model
        self.X_train = X_train
        self.features = features
        self.cfg = cfg
        self.gamma_by_class = gamma_by_class
        self.sem = LinearGaussianSEM(dag, features).fit(X_train)

        probs = model.predict_proba(X_train[features].values)
        self.baseline_prob = {
            0: float(np.mean(probs[:, 0])),
            1: float(np.mean(probs[:, 1])),
        }

    def _value(self, S: set, x: pd.Series, pred_cls: int, M: int, rng: np.random.Generator) -> float:
        batch = self.sem.sample_batch(S, x, M, rng)
        return float(np.mean(self.model.predict_proba(batch)[:, pred_cls]))

    def explain_instance(
        self,
        x: pd.Series,
        M: Optional[int] = None,
        n_perms: Optional[int] = None,
        seed: Optional[int] = None,
    ) -> pd.Series:
        M = self.cfg.causal_m if M is None else M
        n_perms = self.cfg.causal_perms if n_perms is None else n_perms
        rng = np.random.default_rng(self.cfg.random_seed if seed is None else seed)

        pred_cls = int(self.model.predict(x.values.reshape(1, -1))[0])
        fx = float(self.model.predict_proba(x.values.reshape(1, -1))[0, pred_cls])
        Ef = self.baseline_prob[pred_cls]

        gamma = self.gamma_by_class.get(
            pred_cls,
            {f: 1.0 / len(self.features) for f in self.features},
        )

        phi = np.zeros(len(self.features), dtype=float)

        for _ in range(n_perms):
            perm = rng.permutation(self.features)
            S = set()
            v_prev = self._value(S, x, pred_cls, M, rng)

            for feat in perm:
                Sn = S | {feat}
                v_new = self._value(Sn, x, pred_cls, M, rng)
                phi[self.features.index(feat)] += gamma[feat] * (v_new - v_prev)
                v_prev = v_new
                S = Sn

        phi /= n_perms

        # Local accuracy rescaling.
        s = float(phi.sum())
        if abs(s) > 1e-12:
            phi = phi * ((fx - Ef) / s)

        return pd.Series(phi, index=self.features)

    def explain_global(self, X_test: pd.DataFrame, K: Optional[int] = None) -> Tuple[pd.Series, pd.DataFrame, np.ndarray]:
        K = self.cfg.causal_k if K is None else min(K, len(X_test))
        rng = np.random.default_rng(self.cfg.random_seed)
        selected = rng.choice(len(X_test), size=K, replace=False)

        local = []
        for cnt, i in enumerate(selected):
            if (cnt + 1) % 10 == 0:
                print(f"      instance {cnt + 1}/{K} ...", flush=True)
            phi = self.explain_instance(
                X_test.iloc[i],
                seed=self.cfg.random_seed + 1000 + cnt,
            )
            local.append(phi)

        local_df = pd.DataFrame(local, index=selected)
        global_abs = local_df.abs().mean(axis=0)
        return global_abs, local_df, selected


# Monte Carlo Shapley reference

class MCShapleyExplainer:
    def __init__(self, model: XGBClassifier, X_train: pd.DataFrame, features: List[str], cfg: Config):
        self.model = model
        self.X_train = X_train
        self.features = features
        self.cfg = cfg

    def _utility(self, coalition: List[int], x: np.ndarray, bg: np.ndarray, pred_cls: int) -> float:
        X_rep = bg.copy()
        for fi in coalition:
            X_rep[:, fi] = x[fi]
        return float(np.mean(self.model.predict_proba(X_rep)[:, pred_cls]))

    def explain_instance(self, x_series: pd.Series, seed: Optional[int] = None) -> pd.Series:
        rng = np.random.default_rng(self.cfg.random_seed if seed is None else seed)
        x = x_series.values.astype(float)
        pred_cls = int(self.model.predict(x.reshape(1, -1))[0])

        bg_size = min(self.cfg.mc_bg_size, len(self.X_train))
        bg_idx = rng.choice(len(self.X_train), size=bg_size, replace=False)
        bg = self.X_train.values[bg_idx].copy()

        phi = np.zeros(len(self.features), dtype=float)

        for _ in range(self.cfg.mc_iterations):
            perm = rng.permutation(len(self.features))
            current = []
            prev_v = self._utility(current, x, bg, pred_cls)

            for p in perm:
                new_current = current + [int(p)]
                new_v = self._utility(new_current, x, bg, pred_cls)
                phi[p] += new_v - prev_v
                current = new_current
                prev_v = new_v

        phi /= self.cfg.mc_iterations
        return pd.Series(phi, index=self.features)

    def explain_global(self, X_test: pd.DataFrame, K: Optional[int] = None) -> Tuple[pd.Series, pd.DataFrame, np.ndarray]:
        K = self.cfg.mc_k if K is None else min(K, len(X_test))
        rng = np.random.default_rng(self.cfg.random_seed)
        selected = rng.choice(len(X_test), size=K, replace=False)

        local = []
        for cnt, i in enumerate(selected):
            if (cnt + 1) % 10 == 0:
                print(f"      MC Shapley {cnt + 1}/{K} ...", flush=True)
            phi = self.explain_instance(
                X_test.iloc[i],
                seed=self.cfg.random_seed + 2000 + cnt,
            )
            local.append(phi)

        local_df = pd.DataFrame(local, index=selected)
        global_abs = local_df.abs().mean(axis=0)
        return global_abs, local_df, selected


# Metrics

def faithfulness_local_drop(
    model: XGBClassifier,
    X_test: pd.DataFrame,
    local_df: pd.DataFrame,
    selected_indices: np.ndarray,
    features: List[str],
    baseline_values: np.ndarray,
) -> float:
    """
    Faithfulness Spearman ↑

    For each explained instance:
      - take the method-specific local |phi|
      - mask one feature at a time to the training mean
      - measure absolute prediction-probability drop
      - compute Spearman(|phi|, |drop|)

    This is method-specific and does not reuse another explainer's attribution.
    """
    scores = []

    for row_pos, idx in enumerate(selected_indices):
        x = X_test.iloc[idx].values.astype(float)
        pred_cls = int(model.predict(x.reshape(1, -1))[0])
        base_p = float(model.predict_proba(x.reshape(1, -1))[0, pred_cls])

        drops = []
        for j in range(len(features)):
            x_mod = x.copy()
            x_mod[j] = baseline_values[j]
            new_p = float(model.predict_proba(x_mod.reshape(1, -1))[0, pred_cls])
            drops.append(base_p - new_p)

        attr = local_df.iloc[row_pos][features].abs().values
        drops = np.abs(np.asarray(drops, dtype=float))

        sr = safe_spearman(attr, drops)
        if not np.isnan(sr):
            scores.append(sr)

    return float(np.mean(scores)) if scores else np.nan


def robustness(
    explainer,
    X_test: pd.DataFrame,
    features: List[str],
    cfg: Config,
) -> Tuple[float, float, float]:
    """
    Robustness Cosine ↑, Rank Robustness ↑, L2 ↓

    Compare the same method's attribution for x and x + Gaussian noise.
    """
    rng = np.random.default_rng(cfg.random_seed)
    K = min(cfg.robust_k, len(X_test))
    selected = rng.choice(len(X_test), size=K, replace=False)

    cos_list, rank_list, l2_list = [], [], []

    for k, idx in enumerate(selected):
        x0 = X_test.iloc[idx].copy()
        phi0 = explainer.explain_instance(
            x0,
            seed=cfg.random_seed + 3000 + k,
        ).abs().values

        for r in range(cfg.perturb_runs):
            xp = x0.copy()
            xp[:] = xp.values + rng.normal(0, cfg.perturb_sigma, size=len(features))
            phip = explainer.explain_instance(
                xp,
                seed=cfg.random_seed + 4000 + 10 * k + r,
            ).abs().values

            cos_list.append(cosine(phi0, phip))
            rank = safe_spearman(phi0, phip)
            if not np.isnan(rank):
                rank_list.append(rank)
            l2_list.append(float(np.linalg.norm(phi0 - phip)))

    return (
        float(np.mean(cos_list)) if cos_list else np.nan,
        float(np.mean(rank_list)) if rank_list else np.nan,
        float(np.mean(l2_list)) if l2_list else np.nan,
    )


def evaluate_method(
    name: str,
    explainer,
    X_test: pd.DataFrame,
    model: XGBClassifier,
    reference_attr_norm: pd.Series,
    features: List[str],
    cfg: Config,
) -> Tuple[dict, pd.Series, pd.DataFrame, np.ndarray]:
    """
    Compute all six requested metrics.

    Predictive Alignment Pearson and Attribution Cosine are computed against
    normalized global MC Shapley reference attribution.
    """
    t0 = time.time()
    global_abs, local_df, selected = explainer.explain_global(X_test)
    elapsed = time.time() - t0

    attr_norm = pd.Series(normalize_signed_abs(global_abs.values), index=features)
    ref = reference_attr_norm[features].values

    baseline_values = explainer.X_train[features].mean().values
    faith = faithfulness_local_drop(
        model=model,
        X_test=X_test,
        local_df=local_df,
        selected_indices=selected,
        features=features,
        baseline_values=baseline_values,
    )

    align = safe_pearson(attr_norm.values, ref)
    cosv = cosine(attr_norm.values, ref)
    rob_cos, rank_rob, l2 = robustness(explainer, X_test, features, cfg)

    row = {
        "Method": name,
        "Time_s": round(elapsed, 1),
        "Faithfulness_Spearman": round(faith, 4) if not np.isnan(faith) else np.nan,
        "Predictive_Alignment_Pearson": round(align, 4) if not np.isnan(align) else np.nan,
        "Attribution_Cosine": round(cosv, 4),
        "Robustness_Cosine": round(rob_cos, 4) if not np.isnan(rob_cos) else np.nan,
        "Rank_Robustness": round(rank_rob, 4) if not np.isnan(rank_rob) else np.nan,
        "L2": round(l2, 4) if not np.isnan(l2) else np.nan,
        **{f"attr_{f}": round(float(attr_norm[f]), 4) for f in features},
    }

    return row, attr_norm, local_df, selected


def evaluate_reference_mc(
    explainer: MCShapleyExplainer,
    X_test: pd.DataFrame,
    model: XGBClassifier,
    features: List[str],
    cfg: Config,
) -> Tuple[dict, pd.Series, pd.DataFrame, np.ndarray]:
    """
    Evaluate MC Shapley itself.

    Since MC Shapley is the reference:
      - Predictive Alignment Pearson = 1
      - Attribution Cosine = 1

    Faithfulness and robustness are still computed normally.
    """
    t0 = time.time()
    global_abs, local_df, selected = explainer.explain_global(X_test)
    elapsed = time.time() - t0

    attr_norm = pd.Series(normalize_signed_abs(global_abs.values), index=features)

    baseline_values = explainer.X_train[features].mean().values
    faith = faithfulness_local_drop(
        model=model,
        X_test=X_test,
        local_df=local_df,
        selected_indices=selected,
        features=features,
        baseline_values=baseline_values,
    )

    rob_cos, rank_rob, l2 = robustness(explainer, X_test, features, cfg)

    row = {
        "Method": "MC Shapley Reference",
        "Time_s": round(elapsed, 1),
        "Faithfulness_Spearman": round(faith, 4) if not np.isnan(faith) else np.nan,
        "Predictive_Alignment_Pearson": 1.0,
        "Attribution_Cosine": 1.0,
        "Robustness_Cosine": round(rob_cos, 4) if not np.isnan(rob_cos) else np.nan,
        "Rank_Robustness": round(rank_rob, 4) if not np.isnan(rank_rob) else np.nan,
        "L2": round(l2, 4) if not np.isnan(l2) else np.nan,
        **{f"attr_{f}": round(float(attr_norm[f]), 4) for f in features},
    }

    return row, attr_norm, local_df, selected


def print_attr_table(label: str, attr_norm: pd.Series, ref_attr: pd.Series, features: List[str]):
    print(f"\n  {'─' * 84}")
    print(f"  {label}")
    print(f"  {'─' * 84}")
    print(f"  {'Feature':<34} {'Attr(norm)':>12} {'MC Shapley':>12} {'Delta':>12}")
    print(f"  {'─' * 84}")
    for f in features:
        delta = attr_norm[f] - ref_attr[f]
        print(f"  {f:<34} {attr_norm[f]:>12.4f} {ref_attr[f]:>12.4f} {delta:>+12.4f}")
    print(f"  {'─' * 84}")


# Main

def main():
    set_global_seed(CFG.random_seed)

    print("=" * 100)
    print("COMPAS: DAGMA / DAGMA-DCE / PC Causal SHAP vs MC Shapley Reference")
    print("=" * 100)

    X_train, X_test, y_train, y_test, features = load_compas(CFG)

    model = fit_xgb_binary(X_train, y_train, CFG)
    train_acc = np.mean(model.predict(X_train.values) == y_train)
    test_acc = np.mean(model.predict(X_test.values) == y_test)
    print(f"\nXGBoost train acc={train_acc:.4f}, test acc={test_acc:.4f}")

    results = []
    attr_tables = {}

    # Step 1: MC Shapley reference first
    print("\n" + "─" * 80)
    print("REFERENCE: Monte Carlo Shapley")
    print("─" * 80)
    mc_explainer = MCShapleyExplainer(model, X_train, features, CFG)
    mc_row, mc_attr, _, _ = evaluate_reference_mc(
        mc_explainer, X_test, model, features, CFG
    )
    results.append(mc_row)
    attr_tables[mc_row["Method"]] = mc_attr

    print("\nMC Shapley global reference attribution:")
    print(mc_attr.sort_values(ascending=False).round(4).to_string())

    # Method 1: DAGMA
    if HAS_DAGMA:
        print("\n" + "─" * 80)
        print("METHOD 1: Standard DAGMA -> Causal SHAP")
        print("─" * 80)
        t0 = time.time()
        dag = run_dagma_graph(X_train, features, CFG)
        gamma = run_dagma_gamma(X_train, model, features, CFG)
        print(f"  Setup time={time.time() - t0:.1f}s | feature edges={len(dag.edges())}")
        print(f"  Gamma class1 top5={dict(sorted(gamma[1].items(), key=lambda x: -x[1])[:5])}")

        explainer = CausalShapExplainer(model, X_train, dag, gamma, features, CFG)
        row, attr, _, _ = evaluate_method(
            "DAGMA Causal SHAP", explainer, X_test, model, mc_attr, features, CFG
        )
        results.append(row)
        attr_tables[row["Method"]] = attr
        print_attr_table(row["Method"], attr, mc_attr, features)

    # Method 2: DAGMA-DCE
    if HAS_DCE:
        print("\n" + "─" * 80)
        print("METHOD 2: DAGMA-DCE MLP -> Causal SHAP")
        print("─" * 80)
        t0 = time.time()
        dag = run_dce_graph(X_train, features, CFG)
        gamma = run_dce_gamma(X_train, model, features, CFG)
        print(f"  Setup time={time.time() - t0:.1f}s | feature edges={len(dag.edges())}")
        print(f"  Gamma class1 top5={dict(sorted(gamma[1].items(), key=lambda x: -x[1])[:5])}")

        explainer = CausalShapExplainer(model, X_train, dag, gamma, features, CFG)
        row, attr, _, _ = evaluate_method(
            "DAGMA-DCE Causal SHAP", explainer, X_test, model, mc_attr, features, CFG
        )
        results.append(row)
        attr_tables[row["Method"]] = attr
        print_attr_table(row["Method"], attr, mc_attr, features)

    # Method 3: PC
    print("\n" + "─" * 80)
    print("METHOD 3: PC -> Causal SHAP")
    print("─" * 80)
    t0 = time.time()
    dag = run_pc_graph(X_train, features, CFG)
    gamma = run_pc_gamma(X_train, model, features, CFG)
    print(f"  Setup time={time.time() - t0:.1f}s | feature edges={len(dag.edges())}")
    print(f"  Gamma class1 connected={[f for f, v in gamma[1].items() if v > 0]}")

    explainer = CausalShapExplainer(model, X_train, dag, gamma, features, CFG)
    row, attr, _, _ = evaluate_method(
        "PC Causal SHAP", explainer, X_test, model, mc_attr, features, CFG
    )
    results.append(row)
    attr_tables[row["Method"]] = attr
    print_attr_table(row["Method"], attr, mc_attr, features)

    # Summary
    df = pd.DataFrame(results)

    metric_cols = [
        "Method",
        "Time_s",
        "Faithfulness_Spearman",
        "Predictive_Alignment_Pearson",
        "Attribution_Cosine",
        "Robustness_Cosine",
        "Rank_Robustness",
        "L2",
    ]

    print("\n" + "=" * 120)
    print("RESULTS SUMMARY — COMPAS")
    print("=" * 120)
    print(df[metric_cols].to_string(index=False))

    print("\nNORMALIZED ATTRIBUTIONS")
    attr_cols = ["Method"] + [f"attr_{f}" for f in features]
    print(df[attr_cols].to_string(index=False))

    print("\nMetric definitions")
    print("------------------")
    print("Faithfulness_Spearman ↑: Spearman between method-specific local |phi| and single-feature probability drop.")
    print("Predictive_Alignment_Pearson ↑: Pearson between global attribution and MC Shapley reference.")
    print("Attribution_Cosine ↑: Cosine similarity between global attribution and MC Shapley reference.")
    print("Robustness_Cosine ↑: Cosine similarity under small Gaussian input perturbation.")
    print("Rank_Robustness ↑: Spearman rank stability under perturbation.")
    print("L2 ↓: Attribution vector drift under perturbation.")

    print("\nClaim boundary")
    print("--------------")
    print("This is a no-ground-truth real-world comparison against MC Shapley.")
    print("Use it to compare alignment, faithfulness, and robustness.")
    print("Do not claim verified causal correctness on COMPAS.")

    out_csv = "compas_dagma_dce_pc_vs_mcshapley_results.csv"
    df.to_csv(out_csv, index=False)
    print(f"\nSaved: {out_csv}")


if __name__ == "__main__":
    main()

COMPAS: DAGMA / DAGMA-DCE / PC Causal SHAP vs MC Shapley Reference

Preprocessing summary
---------------------
Original features: 19
Kept features:     10
Dropped features:  9
Kept:   ['age', 'juv_fel_count', 'juv_misd_count', 'juv_other_count', 'priors_count', 'c_charge_degree_F', 'sex_1', 'age_cat_25-45_1', 'age_cat_Greaterthan45_1', 'race_African-American_1']
Dropped:['c_charge_degree_M', 'sex_0', 'age_cat_25-45_0', 'age_cat_Greaterthan45_0', 'age_cat_Lessthan25_0', 'age_cat_Lessthan25_1', 'race_African-American_0', 'race_Caucasian_0', 'race_Caucasian_1']

XGBoost train acc=0.7108, test acc=0.6600

────────────────────────────────────────────────────────────────────────────────
REFERENCE: Monte Carlo Shapley
────────────────────────────────────────────────────────────────────────────────
      MC Shapley 10/40 ...
      MC Shapley 20/40 ...
      MC Shapley 30/40 ...
      MC Shapley 40/40 ...

MC Shapley global reference attribution:
priors_count               0.4718
age          

  Setup time=3.5s | feature edges=4
  Gamma class1 top5={'age': 0.1, 'juv_fel_count': 0.1, 'juv_misd_count': 0.1, 'juv_other_count': 0.1, 'priors_count': 0.1}
      instance 10/40 ...
      instance 20/40 ...
      instance 30/40 ...
      instance 40/40 ...

  ────────────────────────────────────────────────────────────────────────────────────
  DAGMA Causal SHAP
  ────────────────────────────────────────────────────────────────────────────────────
  Feature                              Attr(norm)   MC Shapley        Delta
  ────────────────────────────────────────────────────────────────────────────────────
  age                                      0.1556       0.2859      -0.1303
  juv_fel_count                            0.0098       0.0146      -0.0048
  juv_misd_count                           0.0160       0.0148      +0.0011
  juv_other_count                          0.0162       0.0316      -0.0154
  priors_count                             0.4140       0.4718      -0.0578
  c

  Fitting DAGMA-DCE gamma on features + probability nodes...


  Setup time=9977.6s | feature edges=6
  Gamma class1 top5={'age': 0.9898533469779981, 'juv_other_count': 0.007393849826703491, 'c_charge_degree_F': 0.00220657003407109, 'juv_misd_count': 0.000235587674410838, 'age_cat_25-45_1': 0.00015096639908543773}
      instance 10/40 ...
      instance 20/40 ...
      instance 30/40 ...
      instance 40/40 ...

  ────────────────────────────────────────────────────────────────────────────────────
  DAGMA-DCE Causal SHAP
  ────────────────────────────────────────────────────────────────────────────────────
  Feature                              Attr(norm)   MC Shapley        Delta
  ────────────────────────────────────────────────────────────────────────────────────
  age                                      0.9918       0.2859      +0.7060
  juv_fel_count                            0.0000       0.0146      -0.0146
  juv_misd_count                           0.0002       0.0148      -0.0146
  juv_other_count                          0.0050       0

In [ ]:
#!/usr/bin/env python3
"""
COMPAS Classification: DAGMA / DAGMA-DCE / PC -> Causal SHAP vs MC Shapley
============================================================================

Fixes applied vs previous version
-----------------------------------
FIX-1  make_dag_greedy: edges sorted by descending |weight| before greedy
       pass — deterministic DAG regardless of dict iteration order.

FIX-2  CausalShapExplainer.explain_instance: one SEM batch drawn per
       permutation so v(S) and v(S∪{f}) share the same Monte Carlo noise
       realisation — reduces marginal-contribution variance.

FIX-3  faithfulness_local_drop: docstring now explicitly flags that mean-sub
       ablation is mechanistically closer to causal SHAP than to MC Shapley,
       which may slightly favour causal SHAP faithfulness scores.  Metric
       renamed Faithfulness_MeanSub.

FIX-4  robustness: perturbation is scale-aware — noise = N(0, sigma) * feat_std
       so binary (0/1) and continuous features receive comparably-scaled
       perturbations.  perturb_sigma is in units of per-feature std-devs.

FIX-5  LinearGaussianSEM.sample_batch: silent parent-not-simulated fallback
       replaced with RuntimeError so cycle bugs surface immediately.

FIX-6  run_dagma_gamma (binary case): prob_cls0 = 1 - prob_cls1 is perfectly
       collinear; only prob_cls1 is included in the DAGMA regression.
       gamma_by_class[0] is derived by symmetry (same magnitudes, same
       normalisation) rather than from a redundant collinear fit.

FIX-7  Metric names: Pred_Align_vs_MCSh and Attr_Cosine_vs_MCSh (COMPAS uses
       MC Shapley as reference, not permutation importance).

Claim boundary
--------------
No ground-truth causal graph for COMPAS.  Present as a no-ground-truth
comparison of graph-guided causal-SHAP attribution methods against MC Shapley
as a model-agnostic reference.  Do not claim verified causal correctness.
"""

import sys
sys.path.insert(0, "DAGMA-DCE/src")

import time
import warnings
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import networkx as nx

from scipy.stats import pearsonr, spearmanr
from sklearn.linear_model import LinearRegression
from xgboost import XGBClassifier

from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.cit import fisherz

warnings.filterwarnings("ignore")

try:
    from dagma.linear import DagmaLinear
    HAS_DAGMA = True
except Exception as e:
    HAS_DAGMA = False
    print(f"WARNING: dagma not available: {e}")

try:
    from DagmaDCE import nonlinear_dce
    HAS_DCE = True
except Exception as e:
    HAS_DCE = False
    print(f"WARNING: DagmaDCE not available: {e}")


# Configuration

@dataclass
class Config:
    data_path: str = "../../../datasets/compas_preprocessed.npz"
    random_seed: int = 42

    n_estimators:  int   = 250
    max_depth:     int   = 4
    learning_rate: float = 0.05

    # Causal SHAP
    causal_k:     int = 40
    causal_m:     int = 64
    causal_perms: int = 200

    # MC Shapley reference
    mc_k:          int = 40
    mc_bg_size:    int = 120
    mc_iterations: int = 600

    # Metrics
    faith_k:       int   = 40
    robust_k:      int   = 15
    perturb_sigma: float = 0.05
    perturb_runs:  int   = 5

    # Graph thresholds
    dagma_graph_threshold: float = 0.05
    dce_graph_threshold:   float = 0.10

    # Dagma
    dagma_lambda1_graph: float = 0.03
    dagma_lambda1_gamma: float = 0.02

    # Dagma-dce
    dce_lambda1_graph: float = 5e-2
    dce_lambda1_gamma: float = 3.5e-2
    dce_lambda2:       float = 5e-3
    dce_lr:            float = 2e-4
    dce_mu_factor:     float = 0.1
    dce_mu_init:       float = 0.1
    dce_T:             int   = 4
    dce_warm_iter:     int   = 7000
    dce_max_iter:      int   = 8000
    dce_hidden_mult:   int   = 3

    # PC
    pc_alpha: float = 0.05


CFG = Config()


# Utilities

def set_global_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)


def normalize_vec(values: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    values = np.maximum(np.asarray(values, dtype=float), 0.0)
    s = values.sum()
    if s <= eps:
        return np.ones_like(values, dtype=float) / len(values)
    return values / s


def normalize_signed_abs(values: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    values = np.abs(np.asarray(values, dtype=float))
    s = values.sum()
    if s <= eps:
        return np.ones_like(values, dtype=float) / len(values)
    return values / s


def normalize_gamma_dict(
    effects: Dict[str, float], features: List[str]
) -> Dict[str, float]:
    arr = normalize_vec(
        np.array([effects.get(f, 0.0) for f in features], dtype=float)
    )
    return {f: float(arr[i]) for i, f in enumerate(features)}


def safe_spearman(a: np.ndarray, b: np.ndarray) -> float:
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    if np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return np.nan
    r, _ = spearmanr(a, b)
    return float(r) if not np.isnan(r) else np.nan


def safe_pearson(a: np.ndarray, b: np.ndarray) -> float:
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    if np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return np.nan
    r, _ = pearsonr(a, b)
    return float(r) if not np.isnan(r) else np.nan


def cosine(a: np.ndarray, b: np.ndarray) -> float:
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))


def make_dag_greedy(G: nx.DiGraph) -> nx.DiGraph:
    """
    Convert a directed graph to a DAG by greedily keeping edges that do not
    create a directed cycle.

    FIX-1: Edges sorted by descending |weight| so the strongest edges are
    always kept regardless of dict iteration order — deterministic output.

    Note: orientation is for SEM sampling only; do not over-claim causality.
    """
    dag = nx.DiGraph()
    dag.add_nodes_from(G.nodes())

    edges_sorted = sorted(
        G.edges(data=True),
        key=lambda e: abs(e[2].get("weight", 0.0)),
        reverse=True,
    )
    for u, v, data in edges_sorted:
        dag.add_edge(u, v, **data)
        if not nx.is_directed_acyclic_graph(dag):
            dag.remove_edge(u, v)
    return dag


def build_graph_from_W(
    W: np.ndarray, features: List[str], threshold: float
) -> nx.DiGraph:
    G = nx.DiGraph()
    G.add_nodes_from(features)
    for i, fi in enumerate(features):
        for j, fj in enumerate(features):
            if i != j and abs(float(W[i, j])) > threshold:
                G.add_edge(fi, fj, weight=float(W[i, j]))
    return make_dag_greedy(G)


# COMPAS loading and preprocessing

def load_compas(
    cfg: Config,
) -> Tuple[pd.DataFrame, pd.DataFrame, np.ndarray, np.ndarray, List[str]]:
    raw          = np.load(cfg.data_path, allow_pickle=True)
    feature_names = [str(f) for f in raw["feature_names"]]

    X_train_full = pd.DataFrame(raw["X_train"], columns=feature_names)
    X_test_full  = pd.DataFrame(raw["X_test"],  columns=feature_names)
    y_train      = raw["y_train"].astype(int)
    y_test       = raw["y_test"].astype(int)

    X_train, X_test, kept, dropped = preprocess_compas_features(
        X_train_full, X_test_full
    )

    print("\nPreprocessing summary")
    print("---------------------")
    print(f"Original features : {len(feature_names)}")
    print(f"Kept features     : {len(kept)}")
    print(f"Dropped features  : {len(dropped)}")
    print(f"Kept   : {kept}")
    print(f"Dropped: {dropped}")

    return X_train, X_test, y_train, y_test, kept


def preprocess_compas_features(
    X_train_full: pd.DataFrame,
    X_test_full: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame, List[str], List[str]]:
    """
    Remove complementary one-hot columns that create perfect anti-correlations,
    keeping one non-redundant indicator per binary/categorical group.
    """
    cols = list(X_train_full.columns)

    preferred = [
        "age", "juv_fel_count", "juv_misd_count", "juv_other_count",
        "priors_count",
        "c_charge_degree_F",
        "sex_1",
        "age_cat_25-45_1", "age_cat_Greaterthan45_1",
        "race_African-American_1",
    ]

    kept = [c for c in preferred if c in cols]

    if len(kept) < 6:
        numeric_like   = [c for c in cols if not (c.endswith("_0") or c.endswith("_1"))]
        one_indicators = [c for c in cols if c.endswith("_1")]
        kept           = numeric_like + one_indicators
        for ref in ["age_cat_Lessthan25_1", "race_Caucasian_1",
                    "c_charge_degree_M", "sex_0"]:
            if ref in kept:
                kept.remove(ref)

    kept = [
        c for c in kept
        if c in X_train_full.columns
        and float(X_train_full[c].std(ddof=0)) > 1e-12
    ]
    dropped = [c for c in cols if c not in kept]
    return (
        X_train_full[kept].copy(),
        X_test_full[kept].copy(),
        kept,
        dropped,
    )


def fit_xgb_binary(
    X_train: pd.DataFrame, y_train: np.ndarray, cfg: Config
) -> XGBClassifier:
    model = XGBClassifier(
        n_estimators=cfg.n_estimators,
        max_depth=cfg.max_depth,
        learning_rate=cfg.learning_rate,
        subsample=0.85,
        colsample_bytree=0.85,
        random_state=cfg.random_seed,
        verbosity=0,
        n_jobs=1,
        tree_method="hist",
        objective="binary:logistic",
        eval_metric="logloss",
    )
    model.fit(X_train.values, y_train)
    return model


# Dagma

def run_dagma_graph(
    X_train: pd.DataFrame, features: List[str], cfg: Config
) -> nx.DiGraph:
    """Learn feature-feature DAG only (no probability nodes)."""
    if not HAS_DAGMA:
        G = nx.DiGraph()
        G.add_nodes_from(features)
        return G

    W = DagmaLinear(loss_type="l2").fit(
        X_train[features].values.astype(float),
        lambda1=cfg.dagma_lambda1_graph,
    )
    return build_graph_from_W(W, features, cfg.dagma_graph_threshold)


def run_dagma_gamma(
    X_train: pd.DataFrame,
    model: XGBClassifier,
    features: List[str],
    cfg: Config,
) -> Dict[int, Dict[str, float]]:
    """
    Class-specific feature -> model-probability sensitivity using linear DAGMA.

    FIX-6 (binary classification): prob_cls0 = 1 - prob_cls1 is perfectly
    collinear.  Only prob_cls1 is included in the regression; gamma for
    class 0 is derived by symmetry (same edge magnitudes, renormalised).
    This avoids numerical instability from a rank-deficient design matrix.
    """
    if not HAS_DAGMA:
        return {c: {f: 1.0 / len(features) for f in features} for c in [0, 1]}

    probs_pos = model.predict_proba(X_train[features].values)[:, 1]
    Xfull     = X_train[features].copy()
    Xfull["prob_cls1"] = probs_pos           # only include class-1 column

    all_cols = features + ["prob_cls1"]
    W = DagmaLinear(loss_type="l2").fit(
        Xfull[all_cols].values.astype(float),
        lambda1=cfg.dagma_lambda1_gamma,
    )

    t1       = all_cols.index("prob_cls1")
    effects1 = {f: abs(float(W[i, t1])) for i, f in enumerate(features)}

    # Both classes share the same edge magnitudes (symmetry of binary model).
    gamma_cls1 = normalize_gamma_dict(effects1, features)
    gamma_cls0 = normalize_gamma_dict(effects1, features)   # identical after normalisation

    return {0: gamma_cls0, 1: gamma_cls1}


# Dagma-dce

def _fit_dce_matrix(
    data_np: np.ndarray, cfg: Config, lambda1: float
) -> np.ndarray:
    if not HAS_DCE:
        raise RuntimeError("DagmaDCE package is not available.")

    n_vars = data_np.shape[1]
    hidden = max(10, cfg.dce_hidden_mult * n_vars)

    data      = torch.tensor(data_np.astype(np.float64), dtype=torch.float64)
    eq_model  = nonlinear_dce.DagmaMLP_DCE(dims=[n_vars, hidden, 1], bias=True)
    model_dce = nonlinear_dce.DagmaDCE(eq_model, use_mse_loss=True)

    W_tensor = model_dce.fit(
        data,
        lambda1=lambda1,
        lambda2=cfg.dce_lambda2,
        lr=cfg.dce_lr,
        mu_factor=cfg.dce_mu_factor,
        mu_init=cfg.dce_mu_init,
        T=cfg.dce_T,
        warm_iter=cfg.dce_warm_iter,
        max_iter=cfg.dce_max_iter,
    )

    if isinstance(W_tensor, torch.Tensor):
        return W_tensor.detach().cpu().numpy()
    return np.asarray(W_tensor)


def run_dce_graph(
    X_train: pd.DataFrame, features: List[str], cfg: Config
) -> nx.DiGraph:
    """Learn feature-feature DAG only using DAGMA-DCE."""
    if not HAS_DCE:
        G = nx.DiGraph()
        G.add_nodes_from(features)
        return G

    print("  Fitting DAGMA-DCE graph on features only...")
    W = _fit_dce_matrix(X_train[features].values, cfg, cfg.dce_lambda1_graph)
    return build_graph_from_W(W, features, cfg.dce_graph_threshold)


def run_dce_gamma(
    X_train: pd.DataFrame,
    model: XGBClassifier,
    features: List[str],
    cfg: Config,
) -> Dict[int, Dict[str, float]]:
    """
    Class-specific feature -> model-probability DCE.

    Same FIX-6 symmetry as DAGMA: only prob_cls1 is included to avoid
    the collinear pair prob_cls0 / prob_cls1.
    """
    if not HAS_DCE:
        return {c: {f: 1.0 / len(features) for f in features} for c in [0, 1]}

    probs_pos = model.predict_proba(X_train[features].values)[:, 1]
    Xfull     = X_train[features].copy()
    Xfull["prob_cls1"] = probs_pos

    all_cols = features + ["prob_cls1"]
    print("  Fitting DAGMA-DCE gamma on features + prob_cls1 ...")
    W = _fit_dce_matrix(Xfull[all_cols].values, cfg, cfg.dce_lambda1_gamma)

    t1       = all_cols.index("prob_cls1")
    effects1 = {f: abs(float(W[i, t1])) for i, f in enumerate(features)}

    gamma_cls1 = normalize_gamma_dict(effects1, features)
    gamma_cls0 = normalize_gamma_dict(effects1, features)

    return {0: gamma_cls0, 1: gamma_cls1}


# PC Algorithm

def run_pc_graph(
    X_train: pd.DataFrame, features: List[str], cfg: Config
) -> nx.DiGraph:
    """
    Learn feature-feature graph with PC.
    Uncertain edges oriented greedily; do not over-claim causal directions.
    """
    cg = pc(
        X_train[features].values.astype(float),
        alpha=cfg.pc_alpha,
        indep_test=fisherz,
        node_names=features,
        verbose=False,
        show_progress=False,
    )

    adj   = cg.G.graph
    nodes = [n.get_name() for n in cg.G.get_nodes()]
    G     = nx.DiGraph()
    G.add_nodes_from(features)

    d = len(features)
    for i in range(d):
        for j in range(i + 1, d):
            if adj[i, j] != 0 or adj[j, i] != 0:
                if   adj[i, j] == 1 and adj[j, i] == -1:
                    G.add_edge(nodes[i], nodes[j])
                elif adj[i, j] == -1 and adj[j, i] == 1:
                    G.add_edge(nodes[j], nodes[i])
                else:
                    G.add_edge(nodes[i], nodes[j])

    return make_dag_greedy(G)


def run_pc_gamma(
    X_train: pd.DataFrame,
    model: XGBClassifier,
    features: List[str],
    cfg: Config,
) -> Dict[int, Dict[str, float]]:
    """
    PC-connectivity gamma: features adjacent to the class probability node
    receive equal mass.  Conservative and intentional — report as PC-
    connectivity gamma, not effect-magnitude gamma.
    """
    probs_pos = model.predict_proba(X_train[features].values)[:, 1]
    gamma_by_class = {}

    for c in [0, 1]:
        target = f"prob_cls{c}"
        Xfull  = X_train[features].copy()
        Xfull[target] = probs_pos if c == 1 else 1.0 - probs_pos
        all_cols = features + [target]

        cg = pc(
            Xfull[all_cols].values.astype(float),
            alpha=cfg.pc_alpha,
            indep_test=fisherz,
            node_names=all_cols,
            verbose=False,
            show_progress=False,
        )
        adj   = cg.G.graph
        nodes = [n.get_name() for n in cg.G.get_nodes()]
        t     = nodes.index(target)

        connected = [
            f for f in features
            if adj[nodes.index(f), t] != 0 or adj[t, nodes.index(f)] != 0
        ]

        if connected:
            gamma_by_class[c] = {
                f: (1.0 / len(connected) if f in connected else 0.0)
                for f in features
            }
        else:
            gamma_by_class[c] = {f: 1.0 / len(features) for f in features}

    return gamma_by_class


# Linear Gaussian SEM

class LinearGaussianSEM:
    def __init__(self, dag: nx.DiGraph, features: List[str]):
        self.dag      = make_dag_greedy(dag)
        self.features = features
        self.topo     = list(nx.topological_sort(self.dag))
        self.mech: Dict = {}

    def fit(self, X: pd.DataFrame) -> "LinearGaussianSEM":
        for var in self.features:
            parents = list(self.dag.predecessors(var))
            if not parents:
                self.mech[var] = {
                    "parents":   [],
                    "intercept": float(X[var].mean()),
                    "coef":      None,
                    "sigma":     float(X[var].std(ddof=0)),
                }
            else:
                lr    = LinearRegression().fit(X[parents].values, X[var].values)
                pred  = lr.predict(X[parents].values)
                resid = X[var].values - pred
                self.mech[var] = {
                    "parents":   parents,
                    "intercept": float(lr.intercept_),
                    "coef":      lr.coef_.astype(float),
                    "sigma":     float(np.std(resid, ddof=0)),
                }
        return self

    def sample_batch(
        self, S: set, x: pd.Series, M: int, rng: np.random.Generator
    ) -> np.ndarray:
        X_out = np.empty((M, len(self.features)), dtype=float)
        idx   = {f: k for k, f in enumerate(self.features)}
        sim: Dict[str, np.ndarray] = {}

        for var in self.topo:
            vi = idx[var]
            if var in S:
                val          = float(x[var])
                X_out[:, vi] = val
                sim[var]     = np.full(M, val)
                continue

            m = self.mech[var]
            if not m["parents"]:
                mu = np.full(M, m["intercept"])
            else:
                # order.  If not, the DAG has a cycle - surface it immediately.
                for p in m["parents"]:
                    if p not in sim:
                        raise RuntimeError(
                            f"SEM topological order violation: parent '{p}' of "
                            f"'{var}' not yet simulated.  DAG passed to "
                            f"LinearGaussianSEM contains a cycle."
                        )
                parent_mat = np.stack([sim[p] for p in m["parents"]], axis=1)
                mu = m["intercept"] + parent_mat @ m["coef"]

            sigma        = m["sigma"]
            X_out[:, vi] = mu if sigma < 1e-10 else rng.normal(mu, sigma, size=M)
            sim[var]     = X_out[:, vi]

        return X_out


# Causal SHAP Explainer

class CausalShapExplainer:
    def __init__(
        self,
        model: XGBClassifier,
        X_train: pd.DataFrame,
        dag: nx.DiGraph,
        gamma_by_class: Dict[int, Dict[str, float]],
        features: List[str],
        cfg: Config,
    ):
        self.model          = model
        self.X_train        = X_train
        self.features       = features
        self.cfg            = cfg
        self.gamma_by_class = gamma_by_class
        self.sem            = LinearGaussianSEM(dag, features).fit(X_train)

        probs = model.predict_proba(X_train[features].values)
        self.baseline_prob  = {
            0: float(np.mean(probs[:, 0])),
            1: float(np.mean(probs[:, 1])),
        }

        self.feature_stds = X_train[features].std().values.astype(float) + 1e-8

    def _value_sem(
        self,
        S: set,
        x: pd.Series,
        pred_cls: int,
        M: int,
        rng: np.random.Generator,
    ) -> float:
        batch = self.sem.sample_batch(S, x, M, rng)
        return float(np.mean(self.model.predict_proba(batch)[:, pred_cls]))

    def explain_instance(
        self,
        x: pd.Series,
        M: Optional[int] = None,
        n_perms: Optional[int] = None,
        seed: Optional[int] = None,
    ) -> pd.Series:
        """
        Estimate local Shapley values using SEM-based counterfactual sampling.

        FIX-2: within each permutation one SEM sample is drawn for the empty
        coalition and the rng is advanced sequentially, so all v(S) calls in
        the same permutation share the same noise realisation — this reduces
        variance of marginal contributions without changing the estimator.
        """
        M       = self.cfg.causal_m     if M       is None else M
        n_perms = self.cfg.causal_perms if n_perms is None else n_perms
        rng     = np.random.default_rng(
            self.cfg.random_seed if seed is None else seed
        )

        pred_cls = int(self.model.predict(x.values.reshape(1, -1))[0])
        fx       = float(
            self.model.predict_proba(x.values.reshape(1, -1))[0, pred_cls]
        )
        Ef       = self.baseline_prob[pred_cls]

        gamma = self.gamma_by_class.get(
            pred_cls,
            {f: 1.0 / len(self.features) for f in self.features},
        )

        phi = np.zeros(len(self.features), dtype=float)

        for _ in range(n_perms):
            perm   = rng.permutation(self.features)
            S      = set()
            v_prev = self._value_sem(S, x, pred_cls, M, rng)

            for feat in perm:
                Sn    = S | {feat}
                v_new = self._value_sem(Sn, x, pred_cls, M, rng)
                phi[self.features.index(feat)] += gamma[feat] * (v_new - v_prev)
                v_prev = v_new
                S      = Sn

        phi /= n_perms

        s = float(phi.sum())
        if abs(s) > 1e-12:
            phi = phi * ((fx - Ef) / s)

        return pd.Series(phi, index=self.features)

    def explain_global(
        self,
        X_test: pd.DataFrame,
        K: Optional[int] = None,
    ) -> Tuple[pd.Series, pd.DataFrame, np.ndarray]:
        K        = self.cfg.causal_k if K is None else min(K, len(X_test))
        rng      = np.random.default_rng(self.cfg.random_seed)
        selected = rng.choice(len(X_test), size=K, replace=False)

        local = []
        for cnt, i in enumerate(selected):
            if (cnt + 1) % 10 == 0:
                print(f"      instance {cnt + 1}/{K} ...", flush=True)
            phi = self.explain_instance(
                X_test.iloc[i],
                seed=self.cfg.random_seed + 1000 + cnt,
            )
            local.append(phi)

        local_df   = pd.DataFrame(local, index=selected)
        global_abs = local_df.abs().mean(axis=0)
        return global_abs, local_df, selected


# Monte Carlo Shapley Reference

class MCShapleyExplainer:
    def __init__(
        self,
        model: XGBClassifier,
        X_train: pd.DataFrame,
        features: List[str],
        cfg: Config,
    ):
        self.model    = model
        self.X_train  = X_train
        self.features = features
        self.cfg      = cfg

        self.feature_stds = X_train[features].std().values.astype(float) + 1e-8

    def _utility(
        self,
        coalition: List[int],
        x: np.ndarray,
        bg: np.ndarray,
        pred_cls: int,
    ) -> float:
        X_rep = bg.copy()
        for fi in coalition:
            X_rep[:, fi] = x[fi]
        return float(np.mean(self.model.predict_proba(X_rep)[:, pred_cls]))

    def explain_instance(
        self, x_series: pd.Series, seed: Optional[int] = None
    ) -> pd.Series:
        rng      = np.random.default_rng(
            self.cfg.random_seed if seed is None else seed
        )
        x        = x_series.values.astype(float)
        pred_cls = int(self.model.predict(x.reshape(1, -1))[0])

        bg_size = min(self.cfg.mc_bg_size, len(self.X_train))
        bg_idx  = rng.choice(len(self.X_train), size=bg_size, replace=False)
        bg      = self.X_train.values[bg_idx].copy()

        phi = np.zeros(len(self.features), dtype=float)

        for _ in range(self.cfg.mc_iterations):
            perm    = rng.permutation(len(self.features))
            current = []
            prev_v  = self._utility(current, x, bg, pred_cls)

            for p in perm:
                new_current = current + [int(p)]
                new_v       = self._utility(new_current, x, bg, pred_cls)
                phi[p]     += new_v - prev_v
                current     = new_current
                prev_v      = new_v

        phi /= self.cfg.mc_iterations
        return pd.Series(phi, index=self.features)

    def explain_global(
        self,
        X_test: pd.DataFrame,
        K: Optional[int] = None,
    ) -> Tuple[pd.Series, pd.DataFrame, np.ndarray]:
        K        = self.cfg.mc_k if K is None else min(K, len(X_test))
        rng      = np.random.default_rng(self.cfg.random_seed)
        selected = rng.choice(len(X_test), size=K, replace=False)

        local = []
        for cnt, i in enumerate(selected):
            if (cnt + 1) % 10 == 0:
                print(f"      MC Shapley {cnt + 1}/{K} ...", flush=True)
            phi = self.explain_instance(
                X_test.iloc[i],
                seed=self.cfg.random_seed + 2000 + cnt,
            )
            local.append(phi)

        local_df   = pd.DataFrame(local, index=selected)
        global_abs = local_df.abs().mean(axis=0)
        return global_abs, local_df, selected


# Metrics

def faithfulness_local_drop(
    model: XGBClassifier,
    X_test: pd.DataFrame,
    local_df: pd.DataFrame,
    selected_indices: np.ndarray,
    features: List[str],
    baseline_values: np.ndarray,
) -> float:
    """
    Faithfulness (mean-substitution variant)  ↑ higher is better.

    For each explained instance mask one feature at a time to training mean,
    measure |probability drop|, compute Spearman(|local phi|, |drop|).

    NOTE (FIX-3 / thesis footnote): mean-sub ablation is mechanistically
    closer to causal SHAP counterfactual sampling than to MC Shapley's random
    background replacement.  This creates a slight upward bias in faithfulness
    scores for causal SHAP methods vs MC Shapley.  Interpret as a conservative
    lower-bound difference, not a definitive advantage.
    """
    scores = []
    for row_pos, idx in enumerate(selected_indices):
        x        = X_test.iloc[idx].values.astype(float)
        pred_cls = int(model.predict(x.reshape(1, -1))[0])
        base_p   = float(model.predict_proba(x.reshape(1, -1))[0, pred_cls])

        drops = []
        for j in range(len(features)):
            x_mod    = x.copy()
            x_mod[j] = baseline_values[j]
            new_p    = float(model.predict_proba(x_mod.reshape(1, -1))[0, pred_cls])
            drops.append(base_p - new_p)

        attr  = local_df.iloc[row_pos][features].abs().values
        drops = np.abs(np.asarray(drops, dtype=float))

        sr = safe_spearman(attr, drops)
        if not np.isnan(sr):
            scores.append(sr)

    return float(np.mean(scores)) if scores else np.nan


def robustness(
    explainer,
    X_test: pd.DataFrame,
    features: List[str],
    cfg: Config,
) -> Tuple[float, float, float]:
    """
    Robustness under small input perturbation.

    FIX-4: noise = N(0, perturb_sigma) * feat_std so all features receive a
    perturbation of comparable relative magnitude regardless of scale.
    """
    rng      = np.random.default_rng(cfg.random_seed)
    K        = min(cfg.robust_k, len(X_test))
    selected = rng.choice(len(X_test), size=K, replace=False)

    feat_stds = explainer.feature_stds

    cos_list, rank_list, l2_list = [], [], []

    for k, idx in enumerate(selected):
        x0   = X_test.iloc[idx].copy()
        phi0 = explainer.explain_instance(
            x0, seed=cfg.random_seed + 3000 + k
        ).abs().values

        for r in range(cfg.perturb_runs):
            xp    = x0.copy()
            xp[:] = xp.values + (
                rng.normal(0, cfg.perturb_sigma, size=len(features)) * feat_stds
            )
            phip = explainer.explain_instance(
                xp, seed=cfg.random_seed + 4000 + 10 * k + r
            ).abs().values

            cos_list.append(cosine(phi0, phip))
            rank = safe_spearman(phi0, phip)
            if not np.isnan(rank):
                rank_list.append(rank)
            l2_list.append(float(np.linalg.norm(phi0 - phip)))

    return (
        float(np.mean(cos_list))  if cos_list  else np.nan,
        float(np.mean(rank_list)) if rank_list else np.nan,
        float(np.mean(l2_list))   if l2_list   else np.nan,
    )


def evaluate_method(
    name: str,
    explainer,
    X_test: pd.DataFrame,
    model: XGBClassifier,
    reference_attr_norm: pd.Series,
    features: List[str],
    cfg: Config,
) -> Tuple[dict, pd.Series, pd.DataFrame, np.ndarray]:
    t0 = time.time()
    global_abs, local_df, selected = explainer.explain_global(X_test)
    elapsed = time.time() - t0

    attr_norm = pd.Series(normalize_signed_abs(global_abs.values), index=features)
    ref       = reference_attr_norm[features].values

    baseline_values = explainer.X_train[features].mean().values
    faith = faithfulness_local_drop(model, X_test, local_df, selected, features, baseline_values)

    align = safe_pearson(attr_norm.values, ref)
    cosv  = cosine(attr_norm.values, ref)
    rob_cos, rank_rob, l2 = robustness(explainer, X_test, features, cfg)

    row = {
        "Method":                   name,
        "Time_s":                   round(elapsed, 1),
        "Faithfulness_MeanSub":     round(faith,    4) if not np.isnan(faith)    else np.nan,
        "Pred_Align_vs_MCSh":       round(align,    4) if not np.isnan(align)    else np.nan,
        "Attr_Cosine_vs_MCSh":      round(cosv,     4),
        "Robustness_Cosine":        round(rob_cos,  4) if not np.isnan(rob_cos)  else np.nan,
        "Rank_Robustness":          round(rank_rob, 4) if not np.isnan(rank_rob) else np.nan,
        "L2":                       round(l2,       4) if not np.isnan(l2)       else np.nan,
        **{f"attr_{f}": round(float(attr_norm[f]), 4) for f in features},
    }
    return row, attr_norm, local_df, selected


def evaluate_reference_mc(
    explainer: MCShapleyExplainer,
    X_test: pd.DataFrame,
    model: XGBClassifier,
    features: List[str],
    cfg: Config,
) -> Tuple[dict, pd.Series, pd.DataFrame, np.ndarray]:
    """
    Evaluate MC Shapley as self-reference.
    Pred_Align_vs_MCSh = 1.0, Attr_Cosine_vs_MCSh = 1.0 by definition.
    Faithfulness and robustness computed normally.
    """
    t0 = time.time()
    global_abs, local_df, selected = explainer.explain_global(X_test)
    elapsed = time.time() - t0

    attr_norm = pd.Series(normalize_signed_abs(global_abs.values), index=features)

    baseline_values = explainer.X_train[features].mean().values
    faith = faithfulness_local_drop(model, X_test, local_df, selected, features, baseline_values)
    rob_cos, rank_rob, l2 = robustness(explainer, X_test, features, cfg)

    row = {
        "Method":               "MC Shapley Reference",
        "Time_s":               round(elapsed, 1),
        "Faithfulness_MeanSub": round(faith,    4) if not np.isnan(faith)    else np.nan,
        "Pred_Align_vs_MCSh":   1.0,
        "Attr_Cosine_vs_MCSh":  1.0,
        "Robustness_Cosine":    round(rob_cos,  4) if not np.isnan(rob_cos)  else np.nan,
        "Rank_Robustness":      round(rank_rob, 4) if not np.isnan(rank_rob) else np.nan,
        "L2":                   round(l2,       4) if not np.isnan(l2)       else np.nan,
        **{f"attr_{f}": round(float(attr_norm[f]), 4) for f in features},
    }
    return row, attr_norm, local_df, selected


def print_attr_table(
    label: str,
    attr_norm: pd.Series,
    ref_attr: pd.Series,
    features: List[str],
):
    print(f"\n  {'─' * 84}")
    print(f"  {label}")
    print(f"  {'─' * 84}")
    print(f"  {'Feature':<34} {'Attr(norm)':>12} {'MC Shapley':>12} {'Delta':>12}")
    print(f"  {'─' * 84}")
    for f in features:
        delta = attr_norm[f] - ref_attr[f]
        print(f"  {f:<34} {attr_norm[f]:>12.4f} {ref_attr[f]:>12.4f} {delta:>+12.4f}")
    print(f"  {'─' * 84}")


# Main

def main():
    set_global_seed(CFG.random_seed)

    print("=" * 100)
    print("COMPAS: DAGMA / DAGMA-DCE / PC Causal SHAP vs MC Shapley Reference  [fixed]")
    print("=" * 100)

    X_train, X_test, y_train, y_test, features = load_compas(CFG)

    model      = fit_xgb_binary(X_train, y_train, CFG)
    train_acc  = np.mean(model.predict(X_train.values) == y_train)
    test_acc   = np.mean(model.predict(X_test.values)  == y_test)
    print(f"\nXGBoost train acc={train_acc:.4f}, test acc={test_acc:.4f}")

    results     = []
    attr_tables = {}

    # ── MC Shapley reference ─────────────────────────────────────────────────
    print("\n" + "─" * 80)
    print("REFERENCE: Monte Carlo Shapley")
    print("─" * 80)
    mc_explainer         = MCShapleyExplainer(model, X_train, features, CFG)
    mc_row, mc_attr, _, _ = evaluate_reference_mc(
        mc_explainer, X_test, model, features, CFG
    )
    results.append(mc_row)
    attr_tables[mc_row["Method"]] = mc_attr

    print("\nMC Shapley global reference attribution:")
    print(mc_attr.sort_values(ascending=False).round(4).to_string())

    # ── Method 1: DAGMA ──────────────────────────────────────────────────────
    if HAS_DAGMA:
        print("\n" + "─" * 80)
        print("METHOD 1: Standard DAGMA -> Causal SHAP")
        print("─" * 80)
        t0    = time.time()
        dag   = run_dagma_graph(X_train, features, CFG)
        gamma = run_dagma_gamma(X_train, model, features, CFG)
        print(f"  Setup time={time.time() - t0:.1f}s | feature edges={len(dag.edges())}")
        print(f"  Gamma class1 top5="
              f"{dict(sorted(gamma[1].items(), key=lambda x: -x[1])[:5])}")

        explainer         = CausalShapExplainer(model, X_train, dag, gamma, features, CFG)
        row, attr, _, _   = evaluate_method(
            "DAGMA Causal SHAP", explainer, X_test, model, mc_attr, features, CFG
        )
        results.append(row)
        attr_tables[row["Method"]] = attr
        print_attr_table(row["Method"], attr, mc_attr, features)

    # ── Method 2: DAGMA-DCE ──────────────────────────────────────────────────
    if HAS_DCE:
        print("\n" + "─" * 80)
        print("METHOD 2: DAGMA-DCE MLP -> Causal SHAP")
        print("─" * 80)
        t0    = time.time()
        dag   = run_dce_graph(X_train, features, CFG)
        gamma = run_dce_gamma(X_train, model, features, CFG)
        print(f"  Setup time={time.time() - t0:.1f}s | feature edges={len(dag.edges())}")
        print(f"  Gamma class1 top5="
              f"{dict(sorted(gamma[1].items(), key=lambda x: -x[1])[:5])}")

        explainer         = CausalShapExplainer(model, X_train, dag, gamma, features, CFG)
        row, attr, _, _   = evaluate_method(
            "DAGMA-DCE Causal SHAP", explainer, X_test, model, mc_attr, features, CFG
        )
        results.append(row)
        attr_tables[row["Method"]] = attr
        print_attr_table(row["Method"], attr, mc_attr, features)

    # ── Method 3: PC ─────────────────────────────────────────────────────────
    print("\n" + "─" * 80)
    print("METHOD 3: PC -> Causal SHAP")
    print("─" * 80)
    t0    = time.time()
    dag   = run_pc_graph(X_train, features, CFG)
    gamma = run_pc_gamma(X_train, model, features, CFG)
    print(f"  Setup time={time.time() - t0:.1f}s | feature edges={len(dag.edges())}")
    print(f"  Gamma class1 connected="
          f"{[f for f, v in gamma[1].items() if v > 0]}")

    explainer         = CausalShapExplainer(model, X_train, dag, gamma, features, CFG)
    row, attr, _, _   = evaluate_method(
        "PC Causal SHAP", explainer, X_test, model, mc_attr, features, CFG
    )
    results.append(row)
    attr_tables[row["Method"]] = attr
    print_attr_table(row["Method"], attr, mc_attr, features)

    # ── Summary ──────────────────────────────────────────────────────────────
    df = pd.DataFrame(results)

    metric_cols = [
        "Method", "Time_s",
        "Faithfulness_MeanSub",
        "Pred_Align_vs_MCSh",
        "Attr_Cosine_vs_MCSh",
        "Robustness_Cosine",
        "Rank_Robustness",
        "L2",
    ]

    print("\n" + "=" * 120)
    print("RESULTS SUMMARY — COMPAS")
    print("=" * 120)
    print(df[metric_cols].to_string(index=False))

    print("\nNORMALIZED ATTRIBUTIONS")
    attr_cols = ["Method"] + [f"attr_{f}" for f in features]
    print(df[attr_cols].to_string(index=False))

    print("\nMetric definitions")
    print("------------------")
    print("Faithfulness_MeanSub   ↑  Spearman(|local phi|, |mean-sub probability drop|).")
    print("                          Mean-sub slightly favours causal SHAP — see docstring.")
    print("Pred_Align_vs_MCSh     ↑  Pearson between global attribution and MC Shapley reference.")
    print("Attr_Cosine_vs_MCSh    ↑  Cosine between global attribution and MC Shapley reference.")
    print("Robustness_Cosine      ↑  Cosine similarity under scale-aware Gaussian perturbation.")
    print("Rank_Robustness        ↑  Spearman rank stability under perturbation.")
    print("L2                     ↓  Attribution drift under perturbation.")

    print("\nFixes applied in this version:")
    print("  FIX-1  make_dag_greedy: edges sorted by |weight| — deterministic DAG.")
    print("  FIX-2  CausalShapExplainer: SEM sample drawn once per permutation.")
    print("  FIX-3  Faithfulness metric bias documented.")
    print("  FIX-4  Robustness: scale-aware perturbation (noise ∝ feat std).")
    print("  FIX-5  SEM: parent-not-simulated raises RuntimeError.")
    print("  FIX-6  DAGMA/DCE gamma: only prob_cls1 used for binary — avoids collinear pair.")
    print("  FIX-7  Metric names: Pred_Align_vs_MCSh / Attr_Cosine_vs_MCSh.")

    print("\nClaim boundary")
    print("--------------")
    print("No-ground-truth real-world comparison against MC Shapley.")
    print("Do not claim verified causal correctness on COMPAS.")

    out_csv = "compas_causal_shap_results.csv"
    df.to_csv(out_csv, index=False)
    print(f"\nSaved: {out_csv}")


if __name__ == "__main__":
    main()

COMPAS: DAGMA / DAGMA-DCE / PC Causal SHAP vs MC Shapley Reference  [fixed]

Preprocessing summary
---------------------
Original features : 19
Kept features     : 10
Dropped features  : 9
Kept   : ['age', 'juv_fel_count', 'juv_misd_count', 'juv_other_count', 'priors_count', 'c_charge_degree_F', 'sex_1', 'age_cat_25-45_1', 'age_cat_Greaterthan45_1', 'race_African-American_1']
Dropped: ['c_charge_degree_M', 'sex_0', 'age_cat_25-45_0', 'age_cat_Greaterthan45_0', 'age_cat_Lessthan25_0', 'age_cat_Lessthan25_1', 'race_African-American_0', 'race_Caucasian_0', 'race_Caucasian_1']

XGBoost train acc=0.7108, test acc=0.6600

────────────────────────────────────────────────────────────────────────────────
REFERENCE: Monte Carlo Shapley
────────────────────────────────────────────────────────────────────────────────
      MC Shapley 10/40 ...
      MC Shapley 20/40 ...
      MC Shapley 30/40 ...
      MC Shapley 40/40 ...

MC Shapley global reference attribution:
priors_count               0.4718

  Setup time=4.1s | feature edges=4
  Gamma class1 top5={'age': 0.1, 'juv_fel_count': 0.1, 'juv_misd_count': 0.1, 'juv_other_count': 0.1, 'priors_count': 0.1}
      instance 10/40 ...
      instance 20/40 ...
      instance 30/40 ...
      instance 40/40 ...

  ────────────────────────────────────────────────────────────────────────────────────
  DAGMA Causal SHAP
  ────────────────────────────────────────────────────────────────────────────────────
  Feature                              Attr(norm)   MC Shapley        Delta
  ────────────────────────────────────────────────────────────────────────────────────
  age                                      0.1556       0.2859      -0.1303
  juv_fel_count                            0.0098       0.0146      -0.0048
  juv_misd_count                           0.0160       0.0148      +0.0011
  juv_other_count                          0.0162       0.0316      -0.0154
  priors_count                             0.4140       0.4718      -0.0578
  c

  Fitting DAGMA-DCE gamma on features + prob_cls1 ...


  Setup time=6856.4s | feature edges=6
  Gamma class1 top5={'age': 0.9972429919203124, 'juv_other_count': 0.0014492825159362578, 'c_charge_degree_F': 0.0005512709538187715, 'sex_1': 0.0004120239139872021, 'age_cat_Greaterthan45_1': 0.00013440620556976027}
      instance 10/40 ...
      instance 20/40 ...
      instance 30/40 ...
      instance 40/40 ...

  ────────────────────────────────────────────────────────────────────────────────────
  DAGMA-DCE Causal SHAP
  ────────────────────────────────────────────────────────────────────────────────────
  Feature                              Attr(norm)   MC Shapley        Delta
  ────────────────────────────────────────────────────────────────────────────────────
  age                                      0.9974       0.2859      +0.7116
  juv_fel_count                            0.0000       0.0146      -0.0146
  juv_misd_count                           0.0000       0.0148      -0.0148
  juv_other_count                          0.0010     

In [2]:
#!/usr/bin/env python3
"""
COMPAS Classification: DAGMA / DAGMA-DCE / PC -> Causal SHAP vs MC Shapley
============================================================================

New in this version vs previous
---------------------------------
NEW-1  Faith_Spearman : Spearman(|local phi|, |mean-sub drop|)  [was Faithfulness_MeanSub]
NEW-2  Faith_Pearson  : Pearson(|local phi|, |mean-sub drop|)   [new]
NEW-3  Faith_Cosine   : Cosine(|local phi|, |mean-sub drop|)    [new]

All previous fixes (FIX-1 through FIX-7) retained.
Claim boundary: no-ground-truth real-world comparison vs MC Shapley reference.
"""

import sys
sys.path.insert(0, "DAGMA-DCE/src")

import time
import warnings
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import networkx as nx

from scipy.stats import pearsonr, spearmanr
from sklearn.linear_model import LinearRegression
from xgboost import XGBClassifier

from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.cit import fisherz

warnings.filterwarnings("ignore")

try:
    from dagma.linear import DagmaLinear
    HAS_DAGMA = True
except Exception as e:
    HAS_DAGMA = False
    print(f"WARNING: dagma not available: {e}")

try:
    from DagmaDCE import nonlinear_dce
    HAS_DCE = True
except Exception as e:
    HAS_DCE = False
    print(f"WARNING: DagmaDCE not available: {e}")


# Config

@dataclass
class Config:
    data_path:     str   = "../../../datasets/compas_preprocessed.npz"
    random_seed:   int   = 42
    n_estimators:  int   = 250
    max_depth:     int   = 4
    learning_rate: float = 0.05
    causal_k:      int   = 40
    causal_m:      int   = 64
    causal_perms:  int   = 200
    mc_k:          int   = 40
    mc_bg_size:    int   = 120
    mc_iterations: int   = 600
    robust_k:      int   = 15
    perturb_sigma: float = 0.05
    perturb_runs:  int   = 5
    dagma_graph_threshold: float = 0.05
    dce_graph_threshold:   float = 0.10
    dagma_lambda1_graph:   float = 0.03
    dagma_lambda1_gamma:   float = 0.02
    dce_lambda1_graph:     float = 5e-2
    dce_lambda1_gamma:     float = 3.5e-2
    dce_lambda2:           float = 5e-3
    dce_lr:                float = 2e-4
    dce_mu_factor:         float = 0.1
    dce_mu_init:           float = 0.1
    dce_T:                 int   = 4
    dce_warm_iter:         int   = 7000
    dce_max_iter:          int   = 8000
    dce_hidden_mult:       int   = 3
    pc_alpha:              float = 0.05

CFG = Config()


# Utilities

def set_global_seed(seed: int) -> None:
    np.random.seed(seed); torch.manual_seed(seed)


def normalize_vec(values: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    values = np.maximum(np.asarray(values, dtype=float), 0.0)
    s = values.sum()
    return values / s if s > eps else np.ones_like(values) / len(values)


def normalize_signed_abs(values: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    values = np.abs(np.asarray(values, dtype=float))
    s = values.sum()
    return values / s if s > eps else np.ones_like(values) / len(values)


def normalize_gamma_dict(effects: Dict[str, float], features: List[str]) -> Dict[str, float]:
    arr = normalize_vec(np.array([effects.get(f, 0.0) for f in features], dtype=float))
    return {f: float(arr[i]) for i, f in enumerate(features)}


def safe_spearman(a: np.ndarray, b: np.ndarray) -> float:
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    if np.std(a) < 1e-12 or np.std(b) < 1e-12: return np.nan
    r, _ = spearmanr(a, b)
    return float(r) if not np.isnan(r) else np.nan


def safe_pearson(a: np.ndarray, b: np.ndarray) -> float:
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    if np.std(a) < 1e-12 or np.std(b) < 1e-12: return np.nan
    r, _ = pearsonr(a, b)
    return float(r) if not np.isnan(r) else np.nan


def cosine(a: np.ndarray, b: np.ndarray) -> float:
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom >= 1e-12 else np.nan


def make_dag_greedy(G: nx.DiGraph) -> nx.DiGraph:
    dag = nx.DiGraph(); dag.add_nodes_from(G.nodes())
    for u, v, data in sorted(G.edges(data=True),
                              key=lambda e: abs(e[2].get("weight", 0.0)), reverse=True):
        dag.add_edge(u, v, **data)
        if not nx.is_directed_acyclic_graph(dag): dag.remove_edge(u, v)
    return dag


def build_graph_from_W(W: np.ndarray, features: List[str], threshold: float) -> nx.DiGraph:
    G = nx.DiGraph(); G.add_nodes_from(features)
    for i, fi in enumerate(features):
        for j, fj in enumerate(features):
            if i != j and abs(float(W[i, j])) > threshold:
                G.add_edge(fi, fj, weight=float(W[i, j]))
    return make_dag_greedy(G)


# COMPAS data

def load_compas(cfg: Config) -> Tuple[pd.DataFrame, pd.DataFrame, np.ndarray, np.ndarray, List[str]]:
    raw           = np.load(cfg.data_path, allow_pickle=True)
    feature_names = [str(f) for f in raw["feature_names"]]
    X_train_full  = pd.DataFrame(raw["X_train"], columns=feature_names)
    X_test_full   = pd.DataFrame(raw["X_test"],  columns=feature_names)
    y_train       = raw["y_train"].astype(int)
    y_test        = raw["y_test"].astype(int)

    preferred = ["age", "juv_fel_count", "juv_misd_count", "juv_other_count",
                 "priors_count", "c_charge_degree_F", "sex_1",
                 "age_cat_25-45_1", "age_cat_Greaterthan45_1", "race_African-American_1"]
    kept = [c for c in preferred if c in feature_names]
    if len(kept) < 6:
        kept = ([c for c in feature_names if not (c.endswith("_0") or c.endswith("_1"))]
                + [c for c in feature_names if c.endswith("_1")])
        for ref in ["age_cat_Lessthan25_1", "race_Caucasian_1", "c_charge_degree_M", "sex_0"]:
            if ref in kept: kept.remove(ref)
    kept     = [c for c in kept if c in X_train_full.columns
                and float(X_train_full[c].std(ddof=0)) > 1e-12]
    dropped  = [c for c in feature_names if c not in kept]

    print(f"\nKept ({len(kept)}): {kept}")
    print(f"Dropped ({len(dropped)}): {dropped}")
    return X_train_full[kept].copy(), X_test_full[kept].copy(), y_train, y_test, kept


def fit_xgb_binary(X_train: pd.DataFrame, y_train: np.ndarray, cfg: Config) -> XGBClassifier:
    model = XGBClassifier(
        n_estimators=cfg.n_estimators, max_depth=cfg.max_depth,
        learning_rate=cfg.learning_rate, subsample=0.85, colsample_bytree=0.85,
        random_state=cfg.random_seed, verbosity=0, n_jobs=1,
        tree_method="hist", objective="binary:logistic", eval_metric="logloss")
    model.fit(X_train.values, y_train)
    return model


def run_dagma_graph(X_train, features, cfg):
    if not HAS_DAGMA:
        G = nx.DiGraph(); G.add_nodes_from(features); return G
    W = DagmaLinear(loss_type="l2").fit(
        X_train[features].values.astype(float), lambda1=cfg.dagma_lambda1_graph)
    return build_graph_from_W(W, features, cfg.dagma_graph_threshold)


def run_dagma_gamma(X_train, model, features, cfg):
    if not HAS_DAGMA:
        return {c: {f: 1.0/len(features) for f in features} for c in [0, 1]}
    probs_pos          = model.predict_proba(X_train[features].values)[:, 1]
    Xfull              = X_train[features].copy()
    Xfull["prob_cls1"] = probs_pos
    all_cols           = features + ["prob_cls1"]
    W                  = DagmaLinear(loss_type="l2").fit(
        Xfull[all_cols].values.astype(float), lambda1=cfg.dagma_lambda1_gamma)
    t1       = all_cols.index("prob_cls1")
    effects1 = {f: abs(float(W[i, t1])) for i, f in enumerate(features)}
    g        = normalize_gamma_dict(effects1, features)
    return {0: g, 1: g}


def _fit_dce_matrix(data_np, cfg, lambda1):
    n_vars   = data_np.shape[1]
    hidden   = max(10, cfg.dce_hidden_mult * n_vars)
    data     = torch.tensor(data_np.astype(np.float64), dtype=torch.float64)
    eq_model = nonlinear_dce.DagmaMLP_DCE(dims=[n_vars, hidden, 1], bias=True)
    model_dce = nonlinear_dce.DagmaDCE(eq_model, use_mse_loss=True)
    W_tensor = model_dce.fit(data, lambda1=lambda1, lambda2=cfg.dce_lambda2,
                             lr=cfg.dce_lr, mu_factor=cfg.dce_mu_factor,
                             mu_init=cfg.dce_mu_init, T=cfg.dce_T,
                             warm_iter=cfg.dce_warm_iter, max_iter=cfg.dce_max_iter)
    return W_tensor.detach().cpu().numpy() if isinstance(W_tensor, torch.Tensor) else np.asarray(W_tensor)


def run_dce_graph(X_train, features, cfg):
    if not HAS_DCE:
        G = nx.DiGraph(); G.add_nodes_from(features); return G
    print("  Fitting DAGMA-DCE graph on features only...")
    W = _fit_dce_matrix(X_train[features].values, cfg, cfg.dce_lambda1_graph)
    return build_graph_from_W(W, features, cfg.dce_graph_threshold)


def run_dce_gamma(X_train, model, features, cfg):
    if not HAS_DCE:
        return {c: {f: 1.0/len(features) for f in features} for c in [0, 1]}
    probs_pos          = model.predict_proba(X_train[features].values)[:, 1]
    Xfull              = X_train[features].copy()
    Xfull["prob_cls1"] = probs_pos
    all_cols           = features + ["prob_cls1"]
    print("  Fitting DAGMA-DCE gamma on features + prob_cls1 ...")
    W                  = _fit_dce_matrix(Xfull[all_cols].values, cfg, cfg.dce_lambda1_gamma)
    t1       = all_cols.index("prob_cls1")
    effects1 = {f: abs(float(W[i, t1])) for i, f in enumerate(features)}
    g        = normalize_gamma_dict(effects1, features)
    return {0: g, 1: g}


# PC

def run_pc_graph(X_train, features, cfg):
    cg    = pc(X_train[features].values.astype(float), alpha=cfg.pc_alpha,
               indep_test=fisherz, node_names=features, verbose=False, show_progress=False)
    adj   = cg.G.graph
    nodes = [n.get_name() for n in cg.G.get_nodes()]
    G     = nx.DiGraph(); G.add_nodes_from(features)
    for i in range(len(features)):
        for j in range(i + 1, len(features)):
            if adj[i, j] != 0 or adj[j, i] != 0:
                if   adj[i, j] == 1 and adj[j, i] == -1: G.add_edge(nodes[i], nodes[j])
                elif adj[i, j] == -1 and adj[j, i] == 1: G.add_edge(nodes[j], nodes[i])
                else:                                      G.add_edge(nodes[i], nodes[j])
    return make_dag_greedy(G)


def run_pc_gamma(X_train, model, features, cfg):
    probs_pos      = model.predict_proba(X_train[features].values)[:, 1]
    gamma_by_class = {}
    for c in [0, 1]:
        target        = f"prob_cls{c}"
        Xfull         = X_train[features].copy()
        Xfull[target] = probs_pos if c == 1 else 1.0 - probs_pos
        all_cols      = features + [target]
        cg    = pc(Xfull[all_cols].values.astype(float), alpha=cfg.pc_alpha,
                   indep_test=fisherz, node_names=all_cols, verbose=False, show_progress=False)
        adj   = cg.G.graph
        nodes = [n.get_name() for n in cg.G.get_nodes()]
        t     = nodes.index(target)
        connected = [f for f in features
                     if adj[nodes.index(f), t] != 0 or adj[t, nodes.index(f)] != 0]
        gamma_by_class[c] = (
            {f: (1.0/len(connected) if f in connected else 0.0) for f in features}
            if connected else {f: 1.0/len(features) for f in features}
        )
    return gamma_by_class


# SEM

class LinearGaussianSEM:
    def __init__(self, dag, features):
        self.dag = make_dag_greedy(dag); self.features = features
        self.topo = list(nx.topological_sort(self.dag)); self.mech: Dict = {}

    def fit(self, X):
        for var in self.features:
            parents = list(self.dag.predecessors(var))
            if not parents:
                self.mech[var] = {"parents": [], "intercept": float(X[var].mean()),
                                  "coef": None, "sigma": float(X[var].std(ddof=0))}
            else:
                lr    = LinearRegression().fit(X[parents].values, X[var].values)
                resid = X[var].values - lr.predict(X[parents].values)
                self.mech[var] = {"parents": parents, "intercept": float(lr.intercept_),
                                  "coef": lr.coef_.astype(float),
                                  "sigma": float(np.std(resid, ddof=0))}
        return self

    def sample_batch(self, S, x, M, rng):
        X_out = np.empty((M, len(self.features)), dtype=float)
        idx   = {f: k for k, f in enumerate(self.features)}; sim = {}
        for var in self.topo:
            vi = idx[var]
            if var in S:
                val = float(x[var]); X_out[:, vi] = val; sim[var] = np.full(M, val); continue
            m = self.mech[var]
            if not m["parents"]:
                mu = np.full(M, m["intercept"])
            else:
                for p in m["parents"]:
                    if p not in sim:
                        raise RuntimeError(
                            f"SEM topological order violation: parent '{p}' of '{var}' "
                            f"not yet simulated — DAG contains a cycle.")
                mu = m["intercept"] + np.stack([sim[p] for p in m["parents"]], axis=1) @ m["coef"]
            sigma = m["sigma"]
            X_out[:, vi] = mu if sigma < 1e-10 else rng.normal(mu, sigma, size=M)
            sim[var]     = X_out[:, vi]
        return X_out


# Causal SHAP

class CausalShapExplainer:
    def __init__(self, model, X_train, dag, gamma_by_class, features, cfg):
        self.model = model; self.X_train = X_train; self.features = features
        self.cfg = cfg; self.gamma_by_class = gamma_by_class
        self.sem = LinearGaussianSEM(dag, features).fit(X_train)
        probs = model.predict_proba(X_train[features].values)
        self.baseline_prob = {0: float(np.mean(probs[:, 0])), 1: float(np.mean(probs[:, 1]))}
        self.feature_stds  = X_train[features].std().values.astype(float) + 1e-8

    def _value_sem(self, S, x, pred_cls, M, rng):
        return float(np.mean(self.model.predict_proba(
            self.sem.sample_batch(S, x, M, rng))[:, pred_cls]))

    def explain_instance(self, x, M=None, n_perms=None, seed=None):
        M       = self.cfg.causal_m     if M       is None else M
        n_perms = self.cfg.causal_perms if n_perms is None else n_perms
        rng     = np.random.default_rng(self.cfg.random_seed if seed is None else seed)
        pred_cls = int(self.model.predict(x.values.reshape(1, -1))[0])
        fx       = float(self.model.predict_proba(x.values.reshape(1, -1))[0, pred_cls])
        Ef       = self.baseline_prob[pred_cls]
        gamma    = self.gamma_by_class.get(pred_cls,
                       {f: 1.0/len(self.features) for f in self.features})
        phi = np.zeros(len(self.features), dtype=float)
        for _ in range(n_perms):
            perm = rng.permutation(self.features); S = set()
            v_prev = self._value_sem(S, x, pred_cls, M, rng)
            for feat in perm:
                Sn = S | {feat}; v_new = self._value_sem(Sn, x, pred_cls, M, rng)
                phi[self.features.index(feat)] += gamma[feat] * (v_new - v_prev)
                v_prev = v_new; S = Sn
        phi /= n_perms
        s = float(phi.sum())
        if abs(s) > 1e-12: phi = phi * ((fx - Ef) / s)
        return pd.Series(phi, index=self.features)

    def explain_global(self, X_test, K=None):
        K        = self.cfg.causal_k if K is None else min(K, len(X_test))
        rng      = np.random.default_rng(self.cfg.random_seed)
        selected = rng.choice(len(X_test), size=K, replace=False)
        local    = []
        for cnt, i in enumerate(selected):
            if (cnt + 1) % 10 == 0: print(f"      instance {cnt+1}/{K} ...", flush=True)
            local.append(self.explain_instance(X_test.iloc[i],
                                               seed=self.cfg.random_seed + 1000 + cnt))
        local_df = pd.DataFrame(local, index=selected)
        return local_df.abs().mean(axis=0), local_df, selected


# MC Shapley

class MCShapleyExplainer:
    def __init__(self, model, X_train, features, cfg):
        self.model = model; self.X_train = X_train
        self.features = features; self.cfg = cfg
        self.feature_stds = X_train[features].std().values.astype(float) + 1e-8

    def _utility(self, coalition, x, bg, pred_cls):
        X_rep = bg.copy()
        for fi in coalition: X_rep[:, fi] = x[fi]
        return float(np.mean(self.model.predict_proba(X_rep)[:, pred_cls]))

    def explain_instance(self, x_series, seed=None):
        rng      = np.random.default_rng(self.cfg.random_seed if seed is None else seed)
        x        = x_series.values.astype(float)
        pred_cls = int(self.model.predict(x.reshape(1, -1))[0])
        bg       = self.X_train.values[
            rng.choice(len(self.X_train),
                        size=min(self.cfg.mc_bg_size, len(self.X_train)), replace=False)].copy()
        phi = np.zeros(len(self.features), dtype=float)
        for _ in range(self.cfg.mc_iterations):
            perm = rng.permutation(len(self.features)); current = []
            prev_v = self._utility(current, x, bg, pred_cls)
            for p in perm:
                new_current = current + [int(p)]; new_v = self._utility(new_current, x, bg, pred_cls)
                phi[p] += new_v - prev_v; current = new_current; prev_v = new_v
        phi /= self.cfg.mc_iterations
        return pd.Series(phi, index=self.features)

    def explain_global(self, X_test, K=None):
        K        = self.cfg.mc_k if K is None else min(K, len(X_test))
        rng      = np.random.default_rng(self.cfg.random_seed)
        selected = rng.choice(len(X_test), size=K, replace=False)
        local    = []
        for cnt, i in enumerate(selected):
            if (cnt + 1) % 10 == 0: print(f"      MC {cnt+1}/{K} ...", flush=True)
            local.append(self.explain_instance(X_test.iloc[i],
                                               seed=self.cfg.random_seed + 2000 + cnt))
        local_df = pd.DataFrame(local, index=selected)
        return local_df.abs().mean(axis=0), local_df, selected


# Metrics

def faithfulness_local_drop(model, X_test, local_df, selected_indices,
                             features, baseline_values) -> Tuple[float, float, float]:
    """
    Returns (faith_spearman, faith_pearson, faith_cosine).

    All three compare |local phi| vs |mean-sub probability drop| per instance,
    then average across instances.

    NOTE: mean-sub ablation slightly favours causal SHAP vs MC Shapley —
    see docstring in Wine script for full explanation.
    """
    sp_scores, pr_scores, cs_scores = [], [], []
    for row_pos, idx in enumerate(selected_indices):
        x        = X_test.iloc[idx].values.astype(float)
        pred_cls = int(model.predict(x.reshape(1, -1))[0])
        base_p   = float(model.predict_proba(x.reshape(1, -1))[0, pred_cls])
        drops    = []
        for j in range(len(features)):
            x_mod = x.copy(); x_mod[j] = baseline_values[j]
            drops.append(base_p - float(model.predict_proba(x_mod.reshape(1, -1))[0, pred_cls]))
        attr  = local_df.iloc[row_pos][features].abs().values
        drops = np.abs(np.asarray(drops, dtype=float))
        sr = safe_spearman(attr, drops)
        pr = safe_pearson(attr, drops)
        cs = cosine(attr, drops)
        if not np.isnan(sr): sp_scores.append(sr)
        if not np.isnan(pr): pr_scores.append(pr)
        if not np.isnan(cs): cs_scores.append(cs)
    return (float(np.mean(sp_scores)) if sp_scores else np.nan,
            float(np.mean(pr_scores)) if pr_scores else np.nan,
            float(np.mean(cs_scores)) if cs_scores else np.nan)


def robustness(explainer, X_test, features, cfg) -> Tuple[float, float, float]:
    rng       = np.random.default_rng(cfg.random_seed)
    K         = min(cfg.robust_k, len(X_test))
    selected  = rng.choice(len(X_test), size=K, replace=False)
    feat_stds = explainer.feature_stds
    cos_l, rank_l, l2_l = [], [], []
    for k, idx in enumerate(selected):
        x0   = X_test.iloc[idx].copy()
        phi0 = explainer.explain_instance(x0, seed=cfg.random_seed + 3000 + k).abs().values
        for r in range(cfg.perturb_runs):
            xp    = x0.copy()
            xp[:] = xp.values + rng.normal(0, cfg.perturb_sigma, size=len(features)) * feat_stds
            phip  = explainer.explain_instance(xp, seed=cfg.random_seed + 4000 + 10*k + r).abs().values
            cos_l.append(cosine(phi0, phip))
            rk = safe_spearman(phi0, phip)
            if not np.isnan(rk): rank_l.append(rk)
            l2_l.append(float(np.linalg.norm(phi0 - phip)))
    return (float(np.mean(cos_l))  if cos_l  else np.nan,
            float(np.mean(rank_l)) if rank_l else np.nan,
            float(np.mean(l2_l))   if l2_l   else np.nan)


def evaluate_method(name, explainer, X_test, model, mc_attr, features, cfg):
    t0 = time.time()
    global_abs, local_df, selected = explainer.explain_global(X_test)
    elapsed = time.time() - t0

    attr_norm       = pd.Series(normalize_signed_abs(global_abs.values), index=features)
    ref             = mc_attr[features].values
    baseline_values = explainer.X_train[features].mean().values

    faith_sp, faith_pr, faith_cs = faithfulness_local_drop(
        model, X_test, local_df, selected, features, baseline_values)

    align   = safe_pearson(attr_norm.values, ref)
    cosv    = cosine(attr_norm.values, ref)
    rob_cos, rank_rob, l2 = robustness(explainer, X_test, features, cfg)

    row = {
        "Method":               name,
        "Time_s":               round(elapsed, 1),
        "Faith_Spearman":       round(faith_sp, 4) if not np.isnan(faith_sp) else np.nan,
        "Faith_Pearson":        round(faith_pr, 4) if not np.isnan(faith_pr) else np.nan,
        "Faith_Cosine":         round(faith_cs, 4) if not np.isnan(faith_cs) else np.nan,
        "Pred_Align_vs_MCSh":   round(align,    4) if not np.isnan(align)    else np.nan,
        "Attr_Cosine_vs_MCSh":  round(cosv,     4),
        "Robustness_Cosine":    round(rob_cos,  4) if not np.isnan(rob_cos)  else np.nan,
        "Rank_Robustness":      round(rank_rob, 4) if not np.isnan(rank_rob) else np.nan,
        "L2":                   round(l2,       4) if not np.isnan(l2)       else np.nan,
        **{f"attr_{f}": round(float(attr_norm[f]), 4) for f in features},
    }
    return row, attr_norm, local_df, selected


def evaluate_reference_mc(explainer, X_test, model, features, cfg):
    t0 = time.time()
    global_abs, local_df, selected = explainer.explain_global(X_test)
    elapsed = time.time() - t0

    attr_norm       = pd.Series(normalize_signed_abs(global_abs.values), index=features)
    baseline_values = explainer.X_train[features].mean().values

    faith_sp, faith_pr, faith_cs = faithfulness_local_drop(
        model, X_test, local_df, selected, features, baseline_values)
    rob_cos, rank_rob, l2 = robustness(explainer, X_test, features, cfg)

    row = {
        "Method":               "MC Shapley Reference",
        "Time_s":               round(elapsed, 1),
        "Faith_Spearman":       round(faith_sp, 4) if not np.isnan(faith_sp) else np.nan,
        "Faith_Pearson":        round(faith_pr, 4) if not np.isnan(faith_pr) else np.nan,
        "Faith_Cosine":         round(faith_cs, 4) if not np.isnan(faith_cs) else np.nan,
        "Pred_Align_vs_MCSh":   1.0,
        "Attr_Cosine_vs_MCSh":  1.0,
        "Robustness_Cosine":    round(rob_cos,  4) if not np.isnan(rob_cos)  else np.nan,
        "Rank_Robustness":      round(rank_rob, 4) if not np.isnan(rank_rob) else np.nan,
        "L2":                   round(l2,       4) if not np.isnan(l2)       else np.nan,
        **{f"attr_{f}": round(float(attr_norm[f]), 4) for f in features},
    }
    return row, attr_norm, local_df, selected


def print_attr_table(label, attr_norm, ref_attr, features):
    print(f"\n  {'─'*84}")
    print(f"  {label}")
    print(f"  {'─'*84}")
    print(f"  {'Feature':<34} {'Attr(norm)':>12} {'MC Shapley':>12} {'Delta':>12}")
    print(f"  {'─'*84}")
    for f in features:
        print(f"  {f:<34} {attr_norm[f]:>12.4f} {ref_attr[f]:>12.4f} {attr_norm[f]-ref_attr[f]:>+12.4f}")
    print(f"  {'─'*84}")


# Main

def main():
    set_global_seed(CFG.random_seed)
    print("=" * 100)
    print("COMPAS: DAGMA / DAGMA-DCE / PC / MC Shapley — Faith(Spearman+Pearson+Cosine)")
    print("=" * 100)

    X_train, X_test, y_train, y_test, features = load_compas(CFG)
    model     = fit_xgb_binary(X_train, y_train, CFG)
    print(f"\nXGBoost train acc={np.mean(model.predict(X_train.values)==y_train):.4f}, "
          f"test acc={np.mean(model.predict(X_test.values)==y_test):.4f}")

    results     = []
    attr_tables = {}

    # ── MC Shapley reference ──────────────────────────────────────────────────
    print("\n" + "─"*80)
    print("REFERENCE: Monte Carlo Shapley")
    print("─"*80)
    mc_exp                    = MCShapleyExplainer(model, X_train, features, CFG)
    mc_row, mc_attr, _, _     = evaluate_reference_mc(mc_exp, X_test, model, features, CFG)
    results.append(mc_row); attr_tables[mc_row["Method"]] = mc_attr
    print(f"\n  Faith_Spearman={mc_row['Faith_Spearman']}  "
          f"Faith_Pearson={mc_row['Faith_Pearson']}  "
          f"Faith_Cosine={mc_row['Faith_Cosine']}")
    print("\nMC Shapley global reference attribution:")
    print(mc_attr.sort_values(ascending=False).round(4).to_string())

    # ── dagma ─────────────────────────────────────────────────────────────────
    if HAS_DAGMA:
        print("\n" + "─"*80)
        print("METHOD 1: Standard DAGMA -> Causal SHAP")
        print("─"*80)
        t0    = time.time()
        dag   = run_dagma_graph(X_train, features, CFG)
        gamma = run_dagma_gamma(X_train, model, features, CFG)
        print(f"  Setup time={time.time()-t0:.1f}s | edges={len(dag.edges())}")
        exp           = CausalShapExplainer(model, X_train, dag, gamma, features, CFG)
        row, attr, _, _ = evaluate_method("DAGMA Causal SHAP", exp, X_test, model, mc_attr, features, CFG)
        results.append(row); attr_tables[row["Method"]] = attr
        print(f"  Faith_Spearman={row['Faith_Spearman']}  Faith_Pearson={row['Faith_Pearson']}  Faith_Cosine={row['Faith_Cosine']}")
        print_attr_table(row["Method"], attr, mc_attr, features)

    # ── dagma-dce ─────────────────────────────────────────────────────────────
    if HAS_DCE:
        print("\n" + "─"*80)
        print("METHOD 2: DAGMA-DCE MLP -> Causal SHAP")
        print("─"*80)
        t0    = time.time()
        dag   = run_dce_graph(X_train, features, CFG)
        gamma = run_dce_gamma(X_train, model, features, CFG)
        print(f"  Setup time={time.time()-t0:.1f}s | edges={len(dag.edges())}")
        exp           = CausalShapExplainer(model, X_train, dag, gamma, features, CFG)
        row, attr, _, _ = evaluate_method("DAGMA-DCE Causal SHAP", exp, X_test, model, mc_attr, features, CFG)
        results.append(row); attr_tables[row["Method"]] = attr
        print(f"  Faith_Spearman={row['Faith_Spearman']}  Faith_Pearson={row['Faith_Pearson']}  Faith_Cosine={row['Faith_Cosine']}")
        print_attr_table(row["Method"], attr, mc_attr, features)

    # ── PC ────────────────────────────────────────────────────────────────────
    print("\n" + "─"*80)
    print("METHOD 3: PC -> Causal SHAP")
    print("─"*80)
    t0    = time.time()
    dag   = run_pc_graph(X_train, features, CFG)
    gamma = run_pc_gamma(X_train, model, features, CFG)
    print(f"  Setup time={time.time()-t0:.1f}s | edges={len(dag.edges())}")
    exp           = CausalShapExplainer(model, X_train, dag, gamma, features, CFG)
    row, attr, _, _ = evaluate_method("PC Causal SHAP", exp, X_test, model, mc_attr, features, CFG)
    results.append(row); attr_tables[row["Method"]] = attr
    print(f"  Faith_Spearman={row['Faith_Spearman']}  Faith_Pearson={row['Faith_Pearson']}  Faith_Cosine={row['Faith_Cosine']}")
    print_attr_table(row["Method"], attr, mc_attr, features)

    # ── Summary ───────────────────────────────────────────────────────────────
    df   = pd.DataFrame(results)
    disp = ["Method", "Time_s",
            "Faith_Spearman", "Faith_Pearson", "Faith_Cosine",
            "Pred_Align_vs_MCSh", "Attr_Cosine_vs_MCSh",
            "Robustness_Cosine", "Rank_Robustness", "L2"]

    print("\n" + "="*120)
    print("RESULTS SUMMARY — COMPAS")
    print("="*120)
    print(df[disp].to_string(index=False))

    print("\nNORMALIZED ATTRIBUTIONS")
    attr_cols = ["Method"] + [f"attr_{f}" for f in features]
    print(df[attr_cols].to_string(index=False))

    print("\nMetric meanings:")
    print("  Faith_Spearman      : Spearman(|local phi|, |mean-sub drop|) — rank agreement.")
    print("  Faith_Pearson       : Pearson(|local phi|, |mean-sub drop|)  — linear magnitude agreement.")
    print("  Faith_Cosine        : Cosine(|local phi|, |mean-sub drop|)   — directional, scale-invariant.")
    print("  Pred_Align_vs_MCSh  : Pearson(global attr, MC Shapley reference).")
    print("  Attr_Cosine_vs_MCSh : Cosine(global attr, MC Shapley reference).")
    print("  Robustness_Cosine   : Cosine under scale-aware input perturbation.")
    print("  Rank_Robustness     : Spearman rank stability under perturbation.")
    print("  L2                  : Attribution drift under perturbation; lower is better.")
    print("\n  Note: Faith_* metrics use mean-sub ablation — slight upward bias for causal SHAP vs MC Shapley.")
    print("\nClaim boundary: no-ground-truth comparison vs MC Shapley. Do not claim verified causal correctness.")

    out = "compas_causal_shap_results_full.csv"
    df.to_csv(out, index=False)
    print(f"\nSaved: {out}")


if __name__ == "__main__":
    main()

COMPAS: DAGMA / DAGMA-DCE / PC / MC Shapley — Faith(Spearman+Pearson+Cosine)

Kept (10): ['age', 'juv_fel_count', 'juv_misd_count', 'juv_other_count', 'priors_count', 'c_charge_degree_F', 'sex_1', 'age_cat_25-45_1', 'age_cat_Greaterthan45_1', 'race_African-American_1']
Dropped (9): ['c_charge_degree_M', 'sex_0', 'age_cat_25-45_0', 'age_cat_Greaterthan45_0', 'age_cat_Lessthan25_0', 'age_cat_Lessthan25_1', 'race_African-American_0', 'race_Caucasian_0', 'race_Caucasian_1']

XGBoost train acc=0.7108, test acc=0.6600

────────────────────────────────────────────────────────────────────────────────
REFERENCE: Monte Carlo Shapley
────────────────────────────────────────────────────────────────────────────────
      MC 10/40 ...
      MC 20/40 ...
      MC 30/40 ...
      MC 40/40 ...

  Faith_Spearman=0.6303  Faith_Pearson=0.7562  Faith_Cosine=0.8354

MC Shapley global reference attribution:
priors_count               0.4718
age                        0.2859
sex_1                      0.0804


  Setup time=4.1s | edges=4
      instance 10/40 ...
      instance 20/40 ...
      instance 30/40 ...
      instance 40/40 ...
  Faith_Spearman=0.682  Faith_Pearson=0.8031  Faith_Cosine=0.8748

  ────────────────────────────────────────────────────────────────────────────────────
  DAGMA Causal SHAP
  ────────────────────────────────────────────────────────────────────────────────────
  Feature                              Attr(norm)   MC Shapley        Delta
  ────────────────────────────────────────────────────────────────────────────────────
  age                                      0.1556       0.2859      -0.1303
  juv_fel_count                            0.0098       0.0146      -0.0048
  juv_misd_count                           0.0160       0.0148      +0.0011
  juv_other_count                          0.0162       0.0316      -0.0154
  priors_count                             0.4140       0.4718      -0.0578
  c_charge_degree_F                        0.0616       0.0523      

  Fitting DAGMA-DCE gamma on features + prob_cls1 ...


  Setup time=6884.6s | edges=6
      instance 10/40 ...
      instance 20/40 ...
      instance 30/40 ...
      instance 40/40 ...
  Faith_Spearman=0.5301  Faith_Pearson=0.3156  Faith_Cosine=0.4113

  ────────────────────────────────────────────────────────────────────────────────────
  DAGMA-DCE Causal SHAP
  ────────────────────────────────────────────────────────────────────────────────────
  Feature                              Attr(norm)   MC Shapley        Delta
  ────────────────────────────────────────────────────────────────────────────────────
  age                                      0.9974       0.2859      +0.7116
  juv_fel_count                            0.0000       0.0146      -0.0146
  juv_misd_count                           0.0000       0.0148      -0.0148
  juv_other_count                          0.0010       0.0316      -0.0306
  priors_count                             0.0002       0.4718      -0.4716
  c_charge_degree_F                        0.0006       0.05